# Autophagy: aggregate DVP of HEK293 cells with and without bafilomycin A1

HEK293 cells overexpressing α-synuclein (A53T), seeded with α-synuclein preformed fibrils (PFF) or treated with PBS, without or with bafilomycin A1 (BafA1). Fractions of the cells were cut by laser microdissection for Deep Visual Proteomics (DVP) and measured by mass spectrometry; the protein groups come from DIA-NN. This notebook makes the panels of Figure 2, SI2 and SI3.

| Short | Meaning |
|---|---|
| CWO | cells without aggregates |
| CW | cells with aggregates |
| EA | excised aggregates (obs value "Pure Aggregates") |
| RC | rest cells: the cells with aggregates after the aggregate was excised |
| PFF, PBS | seeded with preformed fibrils; PBS control |
| −BafA1, +BafA1 | without, with bafilomycin A1, an inhibitor of the lysosomal V-ATPase |
| log2FC | log2 fold change |
| FDR | false discovery rate, Benjamini-Hochberg |
| CV | coefficient of variation |
| PCA | principal component analysis |
| ANOVA | analysis of variance |
| GSEA, NES | gene set enrichment analysis, normalised enrichment score |

**Inputs:** `input_dir` (DIA-NN `report.pg_matrix.tsv` and `report.pr_matrix.tsv`, `metadata.csv`), the UniProt-to-gene table `src/Uniprot_to_GeneSymbol.csv` (from the alphapepttools documentation), the MSigDB gene-set caches in `src/` (Reactome and GO biological process, retrieved 2026-07-31; GO cellular component, retrieved 2026-09-08; both identical to MSigDB on 2026-10-29), and the GSEA decision files `review.csv` in `gsea_dir/<contrast>/`. All paths are set in the first cell of section 2.

**Outputs:** result tables in `output_dir`; GSEA results in `gsea_dir`; figure panels in `figures/`, written by `save_panel` and assembled in section 16.

# 1. Imports

In [ ]:
# --- standard library ---
import contextlib
import dataclasses
import io
import logging
import os
import re
import sys
import warnings
from collections import Counter
from datetime import date

# --- scientific stack ---
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import f_oneway
from statsmodels.stats.multitest import multipletests

# --- plotting ---
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerTuple
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.transforms import blended_transform_factory
from adjustText import adjust_text

# --- alphapepttools ---
import alphapepttools as at
from alphapepttools.pl.figure import create_figure, label_axes

# --- project modules in src/ ---
# The path is relative to this notebook's folder, which VS Code uses as the
# working directory of a notebook. Importing figure_style sets the matplotlib and
# alphapepttools style of every panel.
sys.path.insert(0, "src")
from figure_style import (ANNOT_FS, AXES_LW, BG_COLOR, DIRECTION_CMAP, DOWN_COLOR,
                          DRAW_SCALE, HIGHLIGHT_COLOR, LABEL_FS, LINE_LW, MARKER_LW,
                          REF_COLOR, THIN_LW, TICK_FS, TITLE_FS, UP_COLOR,
                          compose_figure, figsize, native_pt, save_panel, style_summary)
from gsea_pipeline import (GSEAConfig, load_gene_sets, run_contrast, run_contrast_from_rank,
                           build_review, assert_pass2_selection, report_undecided,
                           plot_contrast, plot_combined)
from dotplot import ttest_dotplot
from preprocessing import flag_contaminants

logging.getLogger("fontTools.subset").setLevel(logging.WARNING)
today = date.today().strftime("%Y%m%d")      # date stamp of the output file names

# 2. Data loading

In [ ]:
# Every path of the notebook, relative to the repository root.
input_dir  = 'data/HEK/DIANN_OptimizedSettings'       # DIA-NN output, metadata.csv
output_dir = 'results/HEK'   # result tables
# GSEA folder: <contrast>/review.csv in it holds the review decisions, an input;
# the GSEA results are written next to it. It is set apart from output_dir, so
# that a new output_dir keeps the decisions.
gsea_dir   = 'gsea/HEK'
var_metadata_path = 'src/Uniprot_to_GeneSymbol.csv'   # alphapepttools documentation dataset
# Panel folders: save_panel writes into them, section 16 assembles them.
FIG2_DIR = "figures/Figure2_HEK"
SI2_DIR  = "figures/SIs/SI2_HEK_QC_AGG"
SI3_DIR  = "figures/SIs/SI3_HEK_BafA1"
os.makedirs(output_dir, exist_ok=True)

# DIA-NN protein group matrix, sample metadata and the UniProt-to-gene table

adata        = at.io.read_pg_table(f"{input_dir}/report.pg_matrix.tsv", search_engine="diann")
obs_metadata = pd.read_csv(f"{input_dir}/metadata.csv", index_col=0)
var_metadata = pd.read_csv(var_metadata_path, index_col=0)

# Sample metadata onto obs. Samples without a metadata row are dropped.
_n = adata.n_obs
adata = at.pp.add_metadata(adata=adata, incoming_metadata=obs_metadata,
                           axis=0, keep_data_shape=False)
print(f"{adata.n_obs} of {_n} samples have a metadata row; {_n - adata.n_obs} dropped")

# Protein.Group, Protein.Names and Genes of the UniProt-to-gene table onto var,
# matched on the protein group ID; protein groups without a row get NaN.
adata.var = adata.var.merge(var_metadata, left_index=True, right_index=True, how="left")

display(adata)
display(adata.obs.head())
display(adata.var.head())

## 2.1 Style, colours and condition order

In [ ]:
# Page grid, fonts, line widths and colour roles come from src/figure_style.py
# (imported in the first cell), shared by all notebooks of the project; printed
# here as a record of the printed sizes.
print(style_summary())

# ---------------------------------------------------------------- CONDITIONS
# Plotting order: PBS -> cells without -> cells with -> excised aggregates ->
# rest cells, -BafA1 before +BafA1 within each.
biological_order = [
    "-BafA1 PBS Cells Without Aggregates",
    "+BafA1 PBS Cells Without Aggregates",
    "-BafA1 PFF Cells Without Aggregates",
    "+BafA1 PFF Cells Without Aggregates",
    "-BafA1 PFF Cells With Aggregates",
    "+BafA1 PFF Cells With Aggregates",
    "-BafA1 PFF Pure Aggregates",
    "+BafA1 PFF Pure Aggregates",
    "-BafA1 PFF Rest Cells",
    "+BafA1 PFF Rest Cells",
]

# Colour of each fraction. The keys are the obs values; "Pure Aggregates" is
# displayed as EA. The first three colours are Paul Tol's colour-blind-safe
# "light" scheme; EA is darker because violet moves toward blue under
# deuteranopia, and its lightness keeps it apart from CWO.
FRACTION_COLORS = {
    "Cells Without Aggregates": "#7fb3d8",   # blue
    "Cells With Aggregates":    "#e8a07a",   # orange
    "Rest Cells":               "#8cc5b2",   # teal
    "Pure Aggregates":          "#8e6ba8",   # violet, EA
}
FRACTION_SHORT = {
    "Cells Without Aggregates": "CWO",
    "Cells With Aggregates":    "CW",
    "Rest Cells":               "RC",
    "Pure Aggregates":          "EA",
}
PBS_GREY = "#999999"          # unseeded cells, in place of the fraction colour

# Colour of each of the ten conditions, for every panel that shows them all:
# hue = fraction (grey for PBS), lightness = BafA1. -BafA1 is the fraction
# colour, +BafA1 the same colour blended BAF_TINT of the way to white.
BAF_TINT = 0.5

def condition_color(cond):
    base = PBS_GREY if "PBS" in cond else next(c for f, c in FRACTION_COLORS.items() if f in cond)
    if "-BafA1" in cond:
        return base
    r, g, b = mcolors.to_rgb(base)
    return mcolors.to_hex((r + (1 - r) * BAF_TINT, g + (1 - g) * BAF_TINT, b + (1 - b) * BAF_TINT))

biological_color_dict = {c: condition_color(c) for c in biological_order}


def short_label(cond):
    """'-BafA1 PFF Pure Aggregates' -> 'PFF EA −Baf'."""
    seed = "PBS" if "PBS" in cond else "PFF"
    frac = next((s for f, s in FRACTION_SHORT.items() if f in cond), cond)
    baf  = "−Baf" if "-BafA1" in cond else "+Baf"
    return f"{seed} {frac} {baf}"

assert set(FRACTION_COLORS) == set(FRACTION_SHORT)

## 2.2 Preprocessing

In [ ]:
# ============================================================
# Preprocessing: every AnnData object the analysis uses, in one cell
# ============================================================

# --- Variables ---
biological       = "class_renamed"
MIN_PROTEINS     = 2000
MAX_MISSING_COND = 0.0   # no missing value allowed within a condition (step 4)

# ------------------------------------------------------------
# 1. QC metrics: protein groups and precursors per sample
# ------------------------------------------------------------
adata.obs["num_proteins"] = np.sum(~np.isnan(adata.X), axis=1)

_pr = pd.read_csv(f"{input_dir}/report.pr_matrix.tsv", sep="\t", index_col=0).T
_pr = _pr.reindex(adata.obs.index).apply(pd.to_numeric, errors="coerce")
adata.obs["num_precursors"] = np.sum(~np.isnan(_pr.values), axis=1)
del _pr

n_start_samples, n_start_proteins = adata.shape

# ------------------------------------------------------------
# 2. Sample filtering: blanks -> fewer than MIN_PROTEINS -> no biological label
# ------------------------------------------------------------
adata = adata[~adata.obs[biological].isin(["Blank", "EvoTip Blank"]), :].copy()
n_after_blanks = adata.n_obs

adata = adata[adata.obs["num_proteins"] >= MIN_PROTEINS, :].copy()
n_after_outliers = adata.n_obs

# A missing label would become a group of its own in the completeness filter
nan_mask = adata.obs[biological].isna()
if nan_mask.any():
    print(f"WARNING: dropping {nan_mask.sum()} samples with NaN in {biological}")
    adata = adata[~nan_mask, :].copy()
n_after_nan = adata.n_obs

bio_counts = adata.obs[biological].value_counts()
print(f"\nBiological groups (column={biological!r}, n={len(bio_counts)}):")
for g, n in bio_counts.items():
    print(f"  {g!r:45s} n={n}")
print()

# ------------------------------------------------------------
# 3. Contaminants: keratins, cRAP-only protein groups, immunoglobulins and a list
#    of skin, serum and laboratory proteins (src/preprocessing.py)
# ------------------------------------------------------------
adata.var["single_gene_name"] = adata.var["genes"].str.split(";").str[0].copy()
adata.var = flag_contaminants(adata.var)

n_contaminants = int(adata.var["is_contaminant"].sum())
adata = adata[:, ~adata.var["is_contaminant"]].copy()
n_after_contaminants = adata.n_vars

# Sample-filtered, all non-contaminant proteins, no completeness filter
adata_sf = adata.copy()
adata_sf.var["median_intensity"] = np.nanmedian(adata_sf.X, axis=0)
adata_sf.var["protein_rank"]     = np.argsort(np.argsort(-adata_sf.var["median_intensity"]))

# ------------------------------------------------------------
# 4. Completeness
#    adata_strict: no missing value in any sample (for the PCA).
#    adata: no missing value in at least one condition. A protein measured in
#    only some conditions is kept; a t-test gives it no p-value when a compared
#    group has fewer than two values. filter_data_completeness with group_column
#    drops a protein that fails in any condition, so each condition is flagged on
#    its own and a protein is kept if any flag passes.
# ------------------------------------------------------------
adata_strict = at.pp.filter_data_completeness(adata.copy(), max_missing=0, action="drop")

_passes = np.zeros(adata.n_vars, dtype=bool)
for _cond in adata.obs[biological].unique():
    _flagged = at.pp.filter_data_completeness(
        adata.copy(), max_missing=MAX_MISSING_COND,
        group_column=biological, groups=[_cond], action="flag",
    )
    _passes |= _flagged.var["passed_threshold_missing_values"].to_numpy()
adata = adata[:, _passes].copy()

# ------------------------------------------------------------
# 5. CV per condition, on linear intensities
# ------------------------------------------------------------
# Sample SD (ddof=1). at.metrics.coefficient_of_variation uses the population
# SD (ddof=0), which at n = 6 gives CVs about 9% lower, so it is not used here.
# A protein with fewer than three values in a condition gets no CV there.
for condition in adata.obs[biological].unique():
    _x = adata[adata.obs[biological] == condition].X
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        _cv = np.nanstd(_x, axis=0, ddof=1) / np.nanmean(_x, axis=0)
    _cv[np.sum(~np.isnan(_x), axis=0) < 3] = np.nan
    adata.var[f"CV_{condition}"] = _cv

# ------------------------------------------------------------
# 6. Log2 copies, log2(x + 1), missing values kept
# ------------------------------------------------------------
adata_log = adata.copy()
sc.pp.log1p(adata_log, base=2)
adata_strict_log = adata_strict.copy()
sc.pp.log1p(adata_strict_log, base=2)

# ------------------------------------------------------------
# 7. PCA on the globally complete log2 data
# ------------------------------------------------------------
adata_pca = adata_strict_log.copy()
at.tl.pca(adata_pca)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("=" * 64)
print("Preprocessing summary")
print("=" * 64)
print(f"Starting:                    {n_start_samples} samples × {n_start_proteins} proteins")
print(f"After blank filter:          {n_after_blanks} samples  (dropped {n_start_samples - n_after_blanks})")
print(f"After <{MIN_PROTEINS}-protein filter: {n_after_outliers} samples  (dropped {n_after_blanks - n_after_outliers})")
print(f"After NaN-{biological} filter: {n_after_nan} samples  (dropped {n_after_outliers - n_after_nan})")
print(f"After contaminant filter:    {n_after_contaminants} proteins  (dropped {n_contaminants})")
print("-" * 64)
print(f"adata_sf          (all non-contaminant):       {adata_sf.n_obs} × {adata_sf.n_vars}")
print(f"adata_strict      (100% complete):             {adata_strict.n_obs} × {adata_strict.n_vars}")
print(f"adata             (complete in ≥1 condition):  {adata.n_obs} × {adata.n_vars}")
print(f"adata_log         (adata → log2):              {adata_log.n_obs} × {adata_log.n_vars}")
print(f"adata_pca         (strict log2 + PCA):         {adata_pca.n_obs} × {adata_pca.n_vars}")
print("=" * 64)

# Dataset for all downstream analyses:
#   adata_log         complete in at least one condition, log2, missing values kept
#   adata_strict_log  complete in every sample, log2, fewer proteins
adata_to_use = adata_log
filter_label = "adata_log"
print(f"adata_to_use      → {filter_label}: {adata_to_use.n_obs} × {adata_to_use.n_vars}")

# 3. Protein groups and precursors per sample (SI2A)

In [ ]:
# SI2A, 3 x 1 units: protein groups (left axis) and precursors (right axis,
# hatched) per condition, with their means in two rows above the boxes. The
# condition labels are horizontal on two lines; rotated by 90 degrees they took
# about 15 mm of the 39.5 mm height.
fig, axm = create_figure(1, 1, figsize=figsize(3, 1))
ax_left  = axm.next()
ax_right = ax_left.twinx()

# twinx() follows rcParams and hides the right spine, which carries the
# precursor scale here; its left spine would be drawn over ax_left's.
ax_right.spines["right"].set_visible(True)
ax_right.spines["left"].set_visible(False)
ax_left.grid(axis="x", visible=False)        # would cut between every box pair

conditions      = biological_order
n_cond          = len(conditions)
positions_left  = np.arange(n_cond) * 2          # protein groups
positions_right = np.arange(n_cond) * 2 + 0.8    # precursors

pg_data = [adata.obs.loc[adata.obs[biological] == c, "num_proteins"].to_numpy() for c in conditions]
pr_data = [adata.obs.loc[adata.obs[biological] == c, "num_precursors"].to_numpy() for c in conditions]

S_DOT  = native_pt(2.2) ** 2     # sample dots, 2.2 pt across in print
JITTER = 0.15                    # largest horizontal offset of a dot, in x units

for ax, data, pos, hatch, marker, seed in (
        (ax_left,  pg_data, positions_left,  None,  "o", 42),
        (ax_right, pr_data, positions_right, "///", "D", 43)):
    bp = ax.boxplot(data, positions=pos, widths=0.6, patch_artist=True,
                    showfliers=False, zorder=2)
    for i, (box, cond) in enumerate(zip(bp["boxes"], conditions)):
        color = biological_color_dict[cond]
        box.set_facecolor(color)
        box.set_edgecolor("black")
        if hatch:
            box.set_hatch(hatch)
        jitter = np.random.default_rng(seed).uniform(-JITTER, JITTER, len(data[i]))
        ax.scatter(pos[i] + jitter, data[i], s=S_DOT, color=color, marker=marker,
                   edgecolors="black", linewidths=MARKER_LW, zorder=3)

# Both axes start at 0. The precursor axis is scaled by the ratio of the overall
# medians, so the two series sit at matching heights.
HEADROOM = 1.1
pg_max = max(d.max() for d in pg_data)
pr_max = max(d.max() for d in pr_data)
ratio  = np.median(np.concatenate(pr_data)) / np.median(np.concatenate(pg_data))
ax_left.set_ylim(0, pg_max * HEADROOM)
ax_right.set_ylim(0, max(pg_max * HEADROOM * ratio, pr_max * HEADROOM))

X_PAD = 0.7
ax_left.set_xlim(positions_left[0] - X_PAD, positions_right[-1] + X_PAD)

# Means above the boxes: protein groups in the lower row (bold), precursors in
# the upper row (bold italic). The offsets are in points, so the two rows keep
# their spacing whatever height the axes get.
GAP = native_pt(1.5)           # axes top to the lower row
ROW = 1.25 * ANNOT_FS          # lower row to upper row
top_left  = blended_transform_factory(ax_left.transData,  ax_left.transAxes)
top_right = blended_transform_factory(ax_right.transData, ax_right.transAxes)
for i in range(n_cond):
    ax_left.annotate(f"{pg_data[i].mean():.0f}", (positions_left[i], 1), xycoords=top_left,
                     xytext=(0, GAP), textcoords="offset points",
                     ha="center", va="bottom", fontsize=ANNOT_FS, fontweight="bold")
    ax_right.annotate(f"{pr_data[i].mean():.0f}", (positions_right[i], 1), xycoords=top_right,
                      xytext=(0, GAP + ROW), textcoords="offset points",
                      ha="center", va="bottom", fontsize=ANNOT_FS, fontweight="bold",
                      style="italic")

ax_left.set_xticks(positions_left + 0.4)
ax_left.set_xticklabels(["\n".join(short_label(c).rsplit(" ", 1)) for c in conditions])
ax_left.set_ylabel("Protein groups")
ax_right.set_ylabel("Precursors")

ax_left.legend(handles=[
    Patch(facecolor=BG_COLOR, edgecolor="black", linewidth=MARKER_LW, label="Protein groups"),
    Patch(facecolor=BG_COLOR, edgecolor="black", linewidth=MARKER_LW, hatch="///", label="Precursors"),
], loc="lower right")

fig.canvas.draw()
assert ax_right.spines["right"].get_visible(), "the precursor axis lost its spine"
assert ax_right.get_yticklabels(), "the precursor axis has no tick labels"

save_panel(fig, "a", "boxplot_pg_precursors", SI2_DIR)

# 4. Abundance rank plot (Figure 2A)

In [ ]:
# Figure 2A, 1 x 1 unit: every protein ranked by its median intensity across all
# samples, with landmark proteins called out.
fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

S_BG = native_pt(0.8) ** 2      # background dots, 0.8 pt across in print
S_HL = native_pt(2.4) ** 2      # called-out proteins, 2.4 pt across

at.pl.scatter(
    ax=ax, data=adata_sf.var, x_column="protein_rank", y_column="median_intensity",
    color=BG_COLOR, scatter_kwargs={"s": S_BG, "linewidths": 0},
)
ax.set_yscale("log")
# The exponent in a 10^n tick label is set at 70 % size and prints below 5 pt,
# so the ticks show the exponent itself and the axis label says log10.
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{np.log10(v):.0f}"))

highlight_genes = [
    # --- high: the disease protein, plus one landmark for scale ---
    'SNCA',                       # disease protein; A53T overexpression puts it near the top
    'ACTB',                       # housekeeping landmark, for scale
    # --- mid ---
    'SQSTM1',                     # central aggrephagy receptor
    'VCP',                        # segregase at the core of the aggregate signature
    'HSPA1A',                     # proteotoxic-stress program
    'FTH1',                       # ferritinophagy cargo
    'APP',                        # top transmembrane receptor in the BafA1 response
    # --- low: shows the depth of the measurement ---
    'NCOA4',                      # ferritinophagy receptor
    'TMEM59',                     # largest BafA1 responder
]

# Match on the split gene list: var["genes"] is semicolon-separated for merged
# protein groups (e.g. "HSPA1A;HSPA1B"), so .isin() on the raw string would miss
# any symbol that is not alone in its group.
_rows, _hl_labels = [], []
for _idx, _raw in adata_sf.var["genes"].fillna("").items():
    _syms = [s.strip() for s in _raw.split(";")]
    _hit  = next((g for g in highlight_genes if g in _syms), None)
    if _hit is not None:
        _rows.append(_idx)
        _hl_labels.append(_hit)

highlight = adata_sf.var.loc[_rows].copy()
highlight["highlight_label"] = _hl_labels   # the wanted symbol, not the group string
print("not found:", [g for g in highlight_genes if g not in set(_hl_labels)])

ax.scatter(highlight["protein_rank"], highlight["median_intensity"], s=S_HL,
           color=HIGHLIGHT_COLOR, edgecolors="black", linewidths=MARKER_LW, zorder=5)

texts = [ax.text(row["protein_rank"], row["median_intensity"], row["highlight_label"],
                 fontsize=ANNOT_FS, ha="left", va="center")
         for _, row in highlight.iterrows()]

adjust_text(
    texts,
    x=adata_sf.var["protein_rank"].values[::5],
    y=adata_sf.var["median_intensity"].values[::5],
    ax=ax,
    arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW),
    expand=(1.5, 1.0),
    force_static=(1.0, 1.5),
    force_text=(0.5, 1.0),
    force_explode=(0.5, 0.8),
)

label_axes(ax, xlabel="Protein rank", ylabel="log10 median intensity")

save_panel(fig, "a", "rank_abundance", FIG2_DIR)

# 5. Coefficient of variation per condition (SI2B)

In [ ]:
# SI2B, 2 x 1 units: CV of every protein within each condition (adata, linear
# intensities), with the condition labels of the protein group panel.
fig, axm = create_figure(1, 1, figsize=figsize(2, 1))
ax = axm.next()
ax.grid(False)        # the 20 % reference line would sit on top of a grid line

cv_columns = [f"CV_{c}" for c in biological_order if f"CV_{c}" in adata.var.columns]
cv_df = adata.var[cv_columns].melt(var_name="Condition", value_name="CV")
cv_df["Condition"] = cv_df["Condition"].str.replace("CV_", "", regex=False)
cv_df = cv_df.dropna()

at.pl.violinplot(
    ax=ax, data=cv_df, grouping_column="Condition", value_column="CV",
    color_dict=biological_color_dict,
)

# The order is read back from the axes: violinplot sets the tick labels from the
# groups it drew, so labels, colours and means stay aligned with the violins.
conditions = [t.get_text() for t in ax.get_xticklabels()]
assert set(conditions) == set(cv_df["Condition"]), (
    "violinplot drew a different set of groups than cv_df holds: "
    f"{sorted(set(conditions) ^ set(cv_df['Condition']))}"
)

# violinplot fills at 50 % opacity; the condition colours are shown at full
# strength, as in the protein group panel.
bodies = [c for c in ax.collections if isinstance(c, mpl.collections.PolyCollection)]
assert len(bodies) == len(conditions), f"{len(bodies)} violin bodies for {len(conditions)} conditions"
for body, cond in zip(bodies, conditions):
    body.set_facecolor(biological_color_dict[cond])

ax.set_ylim(0, cv_df["CV"].quantile(0.99) * 1.1)
ax.axhline(0.2, color=REF_COLOR, linestyle="--", linewidth=AXES_LW, zorder=0)    # CV 20 %
ax.set_xticklabels([short_label(c).replace(" ", "\n") for c in conditions])

# Mean CV per condition, in a row above the violins as in the protein group panel.
GAP = native_pt(1.5)
top = blended_transform_factory(ax.transData, ax.transAxes)
for x, cond in zip(ax.get_xticks(), conditions):
    ax.annotate(f"{cv_df.loc[cv_df['Condition'] == cond, 'CV'].mean():.2f}", (x, 1),
                xycoords=top, xytext=(0, GAP), textcoords="offset points",
                ha="center", va="bottom", fontsize=ANNOT_FS, fontweight="bold")

label_axes(ax, ylabel="CV per protein")

save_panel(fig, "b", "cv_per_condition", SI2_DIR)

# 6. PCA (Figure 2C)

In [ ]:
# Figure 2C, 1 x 1 unit: PCA of the data complete in every sample (adata_pca).
# Colour = fraction (grey for PBS), lightness = BafA1, both from the style cell.
pc1 = adata_pca.obsm["X_pca_obs"][:, 0]
pc2 = adata_pca.obsm["X_pca_obs"][:, 1]
var_explained = adata_pca.uns["variance_pca_obs"]["variance_ratio"]

DOT = native_pt(2.8)            # marker diameter, 2.8 pt in print

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

for condition in biological_order:
    mask = (adata_pca.obs[biological] == condition).to_numpy()
    if mask.any():
        ax.scatter(pc1[mask], pc2[mask], s=DOT ** 2, color=biological_color_dict[condition],
                   edgecolors="black", linewidths=MARKER_LW, zorder=3)

# Headroom so the cluster labels are not clipped (text is not autoscaled)
_y0, _y1 = ax.get_ylim()
ax.set_ylim(_y0, _y1 + 0.15 * (_y1 - _y0))

# Direct labels for the two super-groups, both at the height of the highest
# point so that they line up with each other.
_ea_mask = adata_pca.obs[biological].str.contains("Pure Aggregates").to_numpy()
for _m, _txt in ((_ea_mask, "Excised\naggregates"), (~_ea_mask, "Cellular\nfractions")):
    ax.annotate(_txt, (pc1[_m].mean(), pc2.max()), xytext=(0, native_pt(1.5)),
                textcoords="offset points", ha="center", va="bottom",
                fontsize=ANNOT_FS, fontweight="bold", linespacing=1.2, color="0.3")

# Legend: one entry per fraction, its -BafA1 and +BafA1 colours side by side,
# in the order of the trajectory panel (CWO -> RC -> CW -> EA), then PBS.
_order = [("PFF", "Cells Without Aggregates"), ("PFF", "Rest Cells"),
          ("PFF", "Cells With Aggregates"), ("PFF", "Pure Aggregates"),
          ("PBS", "Cells Without Aggregates")]
_handles, _labels = [], []
for seed, frac in _order:
    _handles.append(tuple(
        Line2D([], [], ls="none", marker="o", markersize=DOT, markeredgewidth=MARKER_LW,
               markeredgecolor="black",
               markerfacecolor=biological_color_dict[f"{b}BafA1 {seed} {frac}"])
        for b in "-+"))
    _labels.append("PBS" if seed == "PBS" else FRACTION_SHORT[frac])

# The legend goes in the empty band between the excised aggregates and the
# cellular fractions, centred on it, so that it covers no point.
_x_gap  = (pc1[_ea_mask].max() + pc1[~_ea_mask].min()) / 2
_x_frac = ax.transAxes.inverted().transform(ax.transData.transform((_x_gap, 0)))[0]
ax.legend(_handles, _labels, handler_map={tuple: HandlerTuple(ndivide=None)},
          title="−/+ BafA1", loc="center", bbox_to_anchor=(_x_frac, 0.45),
          frameon=True, edgecolor="none", framealpha=1)    # white ground hides the grid

label_axes(ax, xlabel=f"PC1 ({var_explained[0] * 100:.1f}%)",
               ylabel=f"PC2 ({var_explained[1] * 100:.1f}%)")

save_panel(fig, "c", "pca_aggregate_density", FIG2_DIR)

# 7. t-tests

In [ ]:
# Welch t-tests (alphapepttools diff_exp_ttest) on adata_to_use, which the
# preprocessing cell sets. log2fc = mean log2 intensity of the first group minus
# that of the second; FDR by Benjamini-Hochberg within each contrast. One table
# per contrast, in ttest_results and in output_dir.
print(f"Using {filter_label}: {adata_to_use.n_obs} samples × {adata_to_use.n_vars} proteins")


comparisons = {
    # --- BafA1 effect within each fraction ---
    "BafA1_effect_PBS": (
        "+BafA1 PBS Cells Without Aggregates",
        "-BafA1 PBS Cells Without Aggregates",
    ),
    "BafA1_effect_PFF_CellsWithout": (
        "+BafA1 PFF Cells Without Aggregates",
        "-BafA1 PFF Cells Without Aggregates",
    ),
    "BafA1_effect_PFF_CellsWith": (
        "+BafA1 PFF Cells With Aggregates",
        "-BafA1 PFF Cells With Aggregates",
    ),
    "BafA1_effect_PFF_PureAggs": (
        "+BafA1 PFF Pure Aggregates",
        "-BafA1 PFF Pure Aggregates",
    ),
    "BafA1_effect_PFF_RestCells": (
        "+BafA1 PFF Rest Cells",
        "-BafA1 PFF Rest Cells",
    ),
    # --- PFF against PBS, within each BafA1 state ---
    "PFF_vs_PBS_minusBafA1": (
        "-BafA1 PFF Cells Without Aggregates",
        "-BafA1 PBS Cells Without Aggregates",
    ),
    "PFF_vs_PBS_plusBafA1": (
        "+BafA1 PFF Cells Without Aggregates",
        "+BafA1 PBS Cells Without Aggregates",
    ),
    # --- EA and CW against CWO ---
    "PureAggs_vs_CellsWithout_minusBafA1": (
        "-BafA1 PFF Pure Aggregates",
        "-BafA1 PFF Cells Without Aggregates",
    ),
    "PureAggs_vs_CellsWithout_plusBafA1": (
        "+BafA1 PFF Pure Aggregates",
        "+BafA1 PFF Cells Without Aggregates",
    ),
    "CellsWith_vs_CellsWithout_minusBafA1": (
        "-BafA1 PFF Cells With Aggregates",
        "-BafA1 PFF Cells Without Aggregates",
    ),
    "CellsWith_vs_CellsWithout_plusBafA1": (
        "+BafA1 PFF Cells With Aggregates",
        "+BafA1 PFF Cells Without Aggregates",
    ),
    # --- EA against CW, RC against CW and CWO, EA against RC ---
    "PureAggs_vs_CellsWith_minusBafA1": (
        "-BafA1 PFF Pure Aggregates",
        "-BafA1 PFF Cells With Aggregates",
    ),
    "PureAggs_vs_CellsWith_plusBafA1": (
        "+BafA1 PFF Pure Aggregates",
        "+BafA1 PFF Cells With Aggregates",
    ),
    "RestCells_vs_CellsWith_minusBafA1": (
        "-BafA1 PFF Rest Cells",
        "-BafA1 PFF Cells With Aggregates",
    ),
    "RestCells_vs_CellsWith_plusBafA1": (
        "+BafA1 PFF Rest Cells",
        "+BafA1 PFF Cells With Aggregates",
    ),
    "RestCells_vs_CellsWithout_minusBafA1": (
        "-BafA1 PFF Rest Cells",
        "-BafA1 PFF Cells Without Aggregates",
    ),
    "RestCells_vs_CellsWithout_plusBafA1": (
        "+BafA1 PFF Rest Cells",
        "+BafA1 PFF Cells Without Aggregates",
    ),
    "PureAggs_vs_RestCells_minusBafA1": (
        "-BafA1 PFF Pure Aggregates",
        "-BafA1 PFF Rest Cells",
    ),
    "PureAggs_vs_RestCells_plusBafA1": (
        "+BafA1 PFF Pure Aggregates",
        "+BafA1 PFF Rest Cells",
    ),
}

ttest_results = {}
for name, (group1, group2) in comparisons.items():
    result = at.tl.diff_exp_ttest(
        adata=adata_to_use,
        between_column=biological,
        comparison=(group1, group2),
    )
    if result is None:
        print(f"{name}: FAILED")
        continue

    # Gene names next to the protein IDs
    result = result.merge(
        adata_to_use.var[["single_gene_name"]],
        left_on="protein", right_index=True, how="left",
    )
    cols = ["protein", "single_gene_name"] + [c for c in result.columns
                                              if c not in ("protein", "single_gene_name")]
    result = result[cols]
    ttest_results[name] = result

    # Printed counts: up / down = FDR < 0.05 and |log2FC| > 1
    n_sig  = (result["fdr"] < 0.05).sum()
    n_up   = ((result["fdr"] < 0.05) & (result["log2fc"] > 1)).sum()
    n_down = ((result["fdr"] < 0.05) & (result["log2fc"] < -1)).sum()
    print(f"{name}: {len(result)} tested, {n_sig} sig (FDR<0.05), {n_up} up, {n_down} down")
    result.to_csv(f"{output_dir}/{today}_ttest_{filter_label}_{name}.csv", index=False)

print(f"\nCSVs saved to {output_dir}")

# 8. Excision purity

## 8.1 Abundance against aggregate enrichment (SI2D)

In [ ]:
# SI2D, 1 x 1 unit: mean cellular abundance against aggregate enrichment
# (EA / RC, -BafA1). If excision picked up proteins in proportion to their
# abundance, the two would correlate.
cellular_conditions = [c for c in biological_order if "Pure Aggregates" not in c]
cell_mask     = adata_to_use.obs[biological].isin(cellular_conditions)
mean_cellular = adata_to_use[cell_mask].to_df().mean(axis=0)     # mean log2 intensity

ea_rc = ttest_results["PureAggs_vs_RestCells_minusBafA1"]
scatter_df = ea_rc[["protein", "single_gene_name", "log2fc", "fdr"]].copy()
scatter_df["mean_cellular"] = scatter_df["protein"].map(mean_cellular)
scatter_df = scatter_df.dropna(subset=["mean_cellular", "log2fc"])
r_pearson  = np.corrcoef(scatter_df["mean_cellular"], scatter_df["log2fc"])[0, 1]

highlights = ["SQSTM1", "SNCA", "FTH1", "APP", "NCOA4", "RPS27A", "H2AC18", "GAPDH"]
S_BG = native_pt(1.0) ** 2      # background dots, 1.0 pt across in print
S_HL = native_pt(2.4) ** 2      # called-out proteins, 2.4 pt across

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()
ax.scatter(scatter_df["mean_cellular"], scatter_df["log2fc"], s=S_BG,
           color=BG_COLOR, alpha=0.5, linewidths=0, zorder=1)   # alpha shows density
ax.axhline(0, color=REF_COLOR, linestyle="--", linewidth=AXES_LW, zorder=2)

hl = scatter_df[scatter_df["single_gene_name"].isin(highlights)].drop_duplicates("single_gene_name")
print("not found:", sorted(set(highlights) - set(hl["single_gene_name"])))
ax.scatter(hl["mean_cellular"], hl["log2fc"], s=S_HL, color=HIGHLIGHT_COLOR,
           edgecolors="black", linewidths=MARKER_LW, zorder=3)
texts = [ax.text(r["mean_cellular"], r["log2fc"], r["single_gene_name"], fontsize=ANNOT_FS)
         for _, r in hl.iterrows()]
adjust_text(texts, ax=ax, x=hl["mean_cellular"].to_numpy(), y=hl["log2fc"].to_numpy(),
            expand=(1.4, 1.6), arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW))

ax.text(0.03, 0.97, f"r = {r_pearson:.2f}".replace("-", "−"), transform=ax.transAxes,
        ha="left", va="top", fontsize=ANNOT_FS)
label_axes(ax, xlabel="Mean cellular abundance (log2)",
               ylabel="EA / RC log2FC (−BafA1)")

save_panel(fig, "d", "abundance_vs_enrichment", SI2_DIR)

## 8.2 Compartment composition of the signal (SI2E)

In [ ]:
# GO cellular component gene sets of MSigDB, cached in
# src/gsea_genesets_gocc_cache.parquet, for the compartment assignment in the next cell.
gocc_actual = (pd.read_parquet("src/gsea_genesets_gocc_cache.parquet")
               .rename(columns={"source": "geneset", "target": "genesymbol"}))
print(f"GO CC terms in the cache: {gocc_actual['geneset'].nunique()}")

In [ ]:
# SI2E, 2 x 1 units: share of the total signal per compartment and condition.
# Each protein is assigned to one compartment, the smallest compartment gene set
# that contains it (the most specific), or "Other" if none does. A compartment's
# share is the summed linear mean intensity of its proteins over the total.
compartment_terms = {
    "Nucleus": ["GOCC_NUCLEAR_BODY", "GOCC_NUCLEAR_CHROMOSOME", "GOCC_NUCLEAR_ENVELOPE",
                "GOCC_NUCLEAR_PROTEIN_CONTAINING_COMPLEX", "GOCC_CHROMATIN",
                "GOCC_RIBONUCLEOPROTEIN_COMPLEX"],
    "Cytoplasm / cytoskeleton": ["GOCC_CYTOPLASMIC_REGION", "GOCC_MICROTUBULE_CYTOSKELETON",
                "GOCC_SUPRAMOLECULAR_COMPLEX", "GOCC_SUPRAMOLECULAR_POLYMER",
                "GOCC_POLYMERIC_CYTOSKELETAL_FIBER", "GOCC_CENTROSOME",
                "GOCC_MICROTUBULE_ORGANIZING_CENTER"],
    "Mitochondria": ["GOCC_MITOCHONDRION"],
    "ER": ["GOCC_ENDOPLASMIC_RETICULUM_LUMEN", "GOCC_ENDOPLASMIC_RETICULUM_PROTEIN_CONTAINING_COMPLEX",
           "GOCC_NUCLEAR_OUTER_MEMBRANE_ENDOPLASMIC_RETICULUM_MEMBRANE_NETWORK"],
    "Plasma membrane": ["GOCC_PLASMA_MEMBRANE_REGION", "GOCC_PLASMA_MEMBRANE_PROTEIN_COMPLEX",
                        "GOCC_ANCHORING_JUNCTION"],
    "Endosome / lysosome": ["GOCC_LYSOSOMAL_LUMEN", "GOCC_ENDOSOME", "GOCC_VESICLE_MEMBRANE"],
    "Golgi": ["GOCC_GOLGI_APPARATUS"],
    "Extracellular": ["GOCC_COLLAGEN_CONTAINING_EXTRACELLULAR_MATRIX", "GOCC_SECRETORY_VESICLE",
                      "GOCC_SECRETORY_GRANULE"],
    "Other": ["GOCC_CATALYTIC_COMPLEX", "GOCC_INTRACELLULAR_PROTEIN_CONTAINING_COMPLEX",
              "GOCC_TRANSFERASE_COMPLEX", "GOCC_ORGANELLE_SUBCOMPARTMENT",
              "GOCC_SYNAPSE", "GOCC_NEURON_PROJECTION"],
}


compartment_genes = {}
compartment_sizes = {}
for label, terms in compartment_terms.items():
    genes = set()
    for term in terms:
        genes.update(gocc_actual[gocc_actual["geneset"] == term]["genesymbol"].values)
    compartment_genes[label] = genes
    compartment_sizes[label] = len(genes)

print("Compartment gene-set sizes:")
for label, genes in compartment_genes.items():
    print(f"  {label:25s}: {len(genes)} genes")

# One compartment per protein: the smallest matching gene set
name_map = adata_to_use.var["single_gene_name"].to_dict()
protein_compartment = {}
for pid, gene in name_map.items():
    matches = [(label, compartment_sizes[label])
               for label, genes in compartment_genes.items() if gene in genes]
    if matches:
        matches.sort(key=lambda x: x[1])
        protein_compartment[pid] = matches[0][0]
    else:
        protein_compartment[pid] = "Other"

counts = Counter(protein_compartment.values())
print(f"\nAssignments: {dict(sorted(counts.items(), key=lambda x: -x[1]))}")

# Composition per condition: linear mean intensity per protein, summed per
# compartment, as a percentage of the total
conditions = biological_order
compartment_order = list(compartment_terms.keys())

comp_pct = pd.DataFrame(index=conditions, columns=compartment_order, dtype=float)

for cond in conditions:
    mask = adata_to_use.obs[biological] == cond
    sub = adata_to_use[mask].to_df()
    mean_log2 = sub.mean(axis=0)
    mean_lin = np.power(2, mean_log2)

    total = 0
    comp_sums = {}
    for comp in compartment_order:
        prots = [p for p, c in protein_compartment.items() if c == comp]
        s = mean_lin[prots].sum()
        comp_sums[comp] = s
        total += s

    for comp in compartment_order:
        comp_pct.loc[cond, comp] = comp_sums[comp] / total * 100

print("\nCompartment composition (% of total signal):")
print(comp_pct.round(2).to_string())

# Source data of SI2E
comp_pct.round(3).to_csv(f"{output_dir}/{today}_compartment_composition.csv")

# --- Stacked bars -------------------------------------------------------
# Compartment colours are greens, yellows, browns and greys: the hues that are
# not already a fraction, a direction or the call-out colour on the same page.
compartment_colors = {
    "Nucleus":                  "#1b7837",   # dark green
    "Cytoplasm / cytoskeleton": "#e6ab02",   # mustard
    "Mitochondria":             "#8c510a",   # brown
    "ER":                       "#a6d96a",   # yellow-green
    "Plasma membrane":          "#dfc27d",   # sand
    "Endosome / lysosome":      "#666666",   # dark grey
    "Golgi":                    "#80803a",   # olive
    "Extracellular":            "#b3b3b3",   # mid grey
    "Other":                    "#e3e3e3",   # light grey
}

# Ten bars with three-line labels, the compartment key below them.
fig, axm = create_figure(1, 1, figsize=figsize(2, 1))
ax = axm.next()
ax.grid(False)

x = np.arange(len(conditions))
bottom = np.zeros(len(conditions))
for comp in compartment_order:
    vals = comp_pct[comp].to_numpy(dtype=float)
    ax.bar(x, vals, bottom=bottom, width=0.8, label=comp,
           color=compartment_colors[comp], edgecolor="white", linewidth=THIN_LW)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels([short_label(c).replace(" ", "\n") for c in conditions])
for tick, cond in zip(ax.get_xticklabels(), conditions):     # the EA columns
    if "Pure Aggregates" in cond:
        tick.set_fontweight("bold")
        tick.set_color(FRACTION_COLORS["Pure Aggregates"])
ax.set_ylim(0, 100)
ax.set_xlim(-0.6, len(conditions) - 0.4)

# Three columns: four would be 98 mm wide, wider than the panel. "outside lower
# center" makes the layout engine reserve the space below the tick labels;
# borderaxespad=0 puts the key on the bottom edge of the panel.
fig.legend(*ax.get_legend_handles_labels(), loc="outside lower center", ncol=3,
           frameon=False, handlelength=1.0, columnspacing=1.0, borderaxespad=0)
label_axes(ax, ylabel="% of total signal")

save_panel(fig, "e", "compartment_composition", SI2_DIR)

## 8.3 Aggregate ruler (Figure 2D)

In [ ]:
# ============================================================
# Aggregate ruler: marker selection and the position of every well
# ============================================================
# A one-dimensional mixing scale for the -BafA1 PFF arm (24 wells, 6 per fraction):
#   0   = mean composition of the CWO wells
#   100 = mean composition of the EA wells
# Markers are the proteins that differ between EA and CWO (Welch t-test on log2
# compositions). A well's position is the median over the markers of
# (x - CWO) / (EA - CWO), in percent.
# CW and RC are placed on the scale built from all CWO and EA wells; they take
# no part in building it. A CWO or EA well cannot be placed on a scale it helped
# define, so each of these 12 wells is placed by leave-one-out (LOO): it is
# dropped, markers and anchor are rebuilt from the other 11 wells, and then it is
# placed. These readings measure the floor (CWO) and the ceiling (EA) of the scale.
#
# Starts from adata_sf: linear intensities, contaminants removed, no completeness
# filter. The 24 wells are selected before the completeness filter, so proteins
# missing only in the +BafA1 or PBS wells are kept.

# compute_ruler() calls diff_exp_ttest 13 times; its INFO lines are silenced
logging.getLogger("alphapepttools.tl.utils").setLevel(logging.WARNING)

RULER_FRACTIONS = {
    "CWO": "-BafA1 PFF Cells Without Aggregates",   # 0 anchor
    "EA":  "-BafA1 PFF Pure Aggregates",            # 100 anchor
    "CW":  "-BafA1 PFF Cells With Aggregates",
    "RC":  "-BafA1 PFF Rest Cells",
}
LO, HI    = "CWO", "EA"
ROW_ORDER = ["CWO", "RC", "CW", "EA"]   # fixed order of the plot rows, not sorted by the result

FDR_MAX_RULER = 0.05
MIN_LOG2FC    = 1.0   # >= 2-fold between anchors, geometric (on log2 compositions)
MIN_ANCHOR_FC = 2.0   # >= 2-fold between anchors, arithmetic (the real denominator)


def compute_ruler(comp, labels, fdr_max=FDR_MAX_RULER,
                  min_log2fc=MIN_LOG2FC, min_anchor_fc=MIN_ANCHOR_FC):
    """Place every well of the four fractions on the CWO -> EA ruler.

    comp   : DataFrame, wells x features, each row renormalised to sum 1 (linear).
    labels : Series of fraction labels, indexed like comp.

    Returns the well positions, the floor and ceiling, the direction split and
    the captured-mass estimates.
    """
    adata_c = ad.AnnData(
        X=np.log2(comp.to_numpy()),
        obs=pd.DataFrame({biological: labels.to_numpy()}, index=comp.index),
        var=pd.DataFrame(index=comp.columns),
    )

    def wells_of(g):
        return comp.index[(labels == RULER_FRACTIONS[g]).to_numpy()]

    def select(keep=None):
        """Welch t-test on log2 compositions, EA vs CWO. log2fc > 0 = EA-enriched."""
        a = adata_c if keep is None else adata_c[keep].copy()
        with contextlib.redirect_stdout(io.StringIO()):
            res = at.tl.diff_exp_ttest(
                a, between_column=biological,
                comparison=(RULER_FRACTIONS[HI], RULER_FRACTIONS[LO]),
            )
        return res[(res["fdr"] < fdr_max) & (res["log2fc"].abs() >= min_log2fc)]

    def anchor(g, exclude=None):
        """Arithmetic mean composition of a fraction, optionally dropping one well."""
        m = (labels == RULER_FRACTIONS[g]).to_numpy()
        if exclude is not None:
            m = m & (comp.index != exclude)
        return comp.loc[m].mean(axis=0)

    def guard(idx, a_lo, a_hi):
        """Drop markers whose arithmetic anchors are closer than min_anchor_fc.
        The t-test bounds the geometric separation, but the denominator of the
        placement formula is the arithmetic one, so it needs its own floor."""
        pair  = pd.concat([a_lo[idx], a_hi[idx]], axis=1)
        ratio = pair.max(axis=1) / pair.min(axis=1).replace(0, np.nan)
        return ratio.index[(ratio >= min_anchor_fc) & ratio.notna()]

    def place(wells, a_lo, a_hi, mk):
        """Position of each well in percent. One protein one vote (median)."""
        x = comp.loc[wells, mk]
        return ((x - a_lo[mk]) / (a_hi[mk] - a_lo[mk])).median(axis=1) * 100

    # --- Scale from all CWO and EA wells: one marker set for CW and RC ----
    sel_full   = select()
    a_lo_full  = anchor(LO)
    a_hi_full  = anchor(HI)
    mk         = guard(sel_full.index, a_lo_full, a_hi_full)
    fc         = sel_full.loc[mk, "log2fc"]
    m_up, m_dn = fc.index[fc > 0], fc.index[fc < 0]

    pos = {g: place(wells_of(g), a_lo_full, a_hi_full, mk) for g in ("CW", "RC")}

    # --- Anchor wells by leave-one-out -------------------------------------
    # The held-out well leaves its own anchor and the marker selection, so its
    # reading does not depend on itself. These readings give the floor and the
    # ceiling: the anchors do not read exactly 0 and 100.
    for g in (LO, HI):
        vals = {}
        for w in wells_of(g):
            keep    = (comp.index != w)
            a_lo_k  = anchor(LO, exclude=w) if g == LO else a_lo_full
            a_hi_k  = anchor(HI, exclude=w) if g == HI else a_hi_full
            mk_k    = guard(select(keep).index, a_lo_k, a_hi_k)
            vals[w] = place([w], a_lo_k, a_hi_k, mk_k).iloc[0]
        pos[g] = pd.Series(vals)

    # --- Direction split, a check and not a correction --------------------
    # Positions from the EA-enriched markers only and from the EA-depleted
    # markers only agree if CW and RC are additive mixtures.
    split = {
        g: {"enriched": place(wells_of(g), a_lo_full, a_hi_full, m_up).mean(),
            "depleted": place(wells_of(g), a_lo_full, a_hi_full, m_dn).mean()}
        for g in ("CW", "RC")
    }

    # f, g: mean positions of CW and RC as fractions; delta: the floor.
    # w_captured: mass fraction of a CW well that the excised aggregate accounts
    # for, from CW = w·EA + (1−w)·RC on the scale, i.e. f = w + (1−w)·g.
    # w_total: aggregate-like fraction of a CW well above the floor.
    f_pct, g_pct = pos["CW"].mean(), pos["RC"].mean()
    floor_pct    = pos[LO].mean()
    f, gg, delta = f_pct / 100, g_pct / 100, floor_pct / 100
    w_wells      = (pos["CW"] / 100 - gg) / (1 - gg)

    return {
        "pos": pos, "split": split,
        "m_enriched": m_up, "m_depleted": m_dn,
        "floor_pct": floor_pct, "ceil_pct": pos[HI].mean(),
        "w_captured": (f - gg) / (1 - gg),            # floor cancels in the numerator
        "w_captured_sd": float(w_wells.std(ddof=1)),
        "w_total": f - delta,                          # floor must be subtracted here
        "w_total_raw": f,
        "n_markers": int(len(mk)), "n_selected": int(len(sel_full)),
    }


# ------------------------------------------------------------
# The 24 wells, then completeness, then compositions
# ------------------------------------------------------------
_in_ruler   = adata_sf.obs[biological].isin(RULER_FRACTIONS.values())
adata_ruler = adata_sf[_in_ruler].copy()
_n_before   = adata_ruler.n_vars

# No group_column: complete across all 24 wells, which the ratio needs.
adata_ruler = at.pp.filter_data_completeness(adata_ruler, max_missing=0, action="drop")
_n_after    = adata_ruler.n_vars

# Every well renormalised to sum 1. A mixture is additive in amount, so this is
# done on linear intensities, after the subset and after the completeness filter.
_lin    = adata_ruler.to_df()
comp    = _lin.div(_lin.sum(axis=1), axis=0)
_labels = adata_ruler.obs[biological]

ruler = compute_ruler(comp, _labels)


def _ratio(w):
    return f"1 : {(1 - w) / w:.1f}" if w > 0 else "n/a"


print("Wells:     " + str(adata_ruler.n_obs) + "  ("
      + ", ".join(f"{k}={int((_labels == v).sum())}" for k, v in RULER_FRACTIONS.items()) + ")")
print(f"Proteins:  {_n_before} -> {_n_after} complete in all {adata_ruler.n_obs} wells "
      f"(dropped {_n_before - _n_after}; EA-exclusive proteins are among them, by design)")
print(f"Markers:   {ruler['n_selected']} pass FDR<{FDR_MAX_RULER} & |log2FC|>={MIN_LOG2FC}"
      f" -> {ruler['n_markers']} also clear the {MIN_ANCHOR_FC}x arithmetic-anchor guard"
      f"  ({len(ruler['m_enriched'])} EA-enriched, {len(ruler['m_depleted'])} EA-depleted)")

print(f"\n{'row':<5}{'mean %':>9}{'SD':>7}   individual wells")
for _g in ROW_ORDER:
    _v = ruler["pos"][_g]
    print(f"{_g:<5}{_v.mean():>9.1f}{_v.std(ddof=1):>7.1f}   "
          + "  ".join(f"{x:.1f}" for x in _v))

print(f"\nFloor   (LOO {LO}): {ruler['floor_pct']:+.1f} %   "
      f"<- what an aggregate-free well actually reads")
print(f"Ceiling (LOO {HI}): {ruler['ceil_pct']:.1f} %")

print("\nDirection split (must agree if the mixture is additive):")
for _g in ("CW", "RC"):
    _s = ruler["split"][_g]
    print(f"  {_g}: enriched-only {_s['enriched']:.1f} %   depleted-only {_s['depleted']:.1f} %"
          f"   (difference {_s['enriched'] - _s['depleted']:+.1f})")

print(f"\nAggregate mass captured by excision    w = (f-g)/(1-g) = "
      f"{ruler['w_captured'] * 100:.1f} % +/- {ruler['w_captured_sd'] * 100:.1f}"
      f"   ->  {_ratio(ruler['w_captured'])}")
print(f"Total aggregate-like mass in the cell  w = f - floor   = "
      f"{ruler['w_total'] * 100:.1f} %   (raw f = {ruler['w_total_raw'] * 100:.1f} %)"
      f"   ->  {_ratio(ruler['w_total'])}")

In [ ]:
# ============================================================
# Figure 2D, 2 x 1 units: the aggregate ruler on a broken x axis
# ============================================================
# Every point is drawn on both segments and clipped by their x limits. The rows
# are in the fixed order of ROW_ORDER (CWO at the bottom, EA at the top), so an
# increase from row to row is a result of the data.
#
# Colour = fraction, from FRACTION_COLORS. Filled = measured on the scale (CW,
# RC); open = leave-one-out control (CWO, EA). Error bars are the SD across the
# six wells: well-to-well reproducibility, not the uncertainty of the estimator.
#
# subplots_adjust sets the margins by hand: the row labels sit outside their
# axes and the x label spans both halves of the broken axis. save_panel reports
# anything these margins push past the canvas edge.

LEFT_XLIM, LEFT_TICKS   = (-4, 21), [0, 5, 10, 15, 20]
RIGHT_XLIM, RIGHT_TICKS = (90, 110), [100]
MARGINS    = dict(left=0.10, right=0.985, top=0.93, bottom=0.20)
XLABEL_X, XLABEL_Y = 0.60, 0.035     # figure fraction; raise XLABEL_Y to tighten the gap
BOTTOM_PAD = 0.4                     # blank space below the CWO row, in row heights

DOT    = native_pt(7.0)              # marker diameter, the same in all four rows
CAP    = native_pt(2.0)              # error-bar cap length
BREAK  = native_pt(5.0)              # axis-break glyph size
MASK   = dict(boxstyle="square,pad=0.1", fc="white", ec="none")   # hides the 0 / 100 lines behind text

Y          = {g: i for i, g in enumerate(ROW_ORDER)}   # CWO bottom -> EA top
CONTROLS   = (LO, HI)

# short code -> fraction key, so colours come from the one central definition
_short_to_fraction = {v: k for k, v in FRACTION_SHORT.items()}
_colors = {g: FRACTION_COLORS[_short_to_fraction[g]] for g in ROW_ORDER}
_pos    = ruler["pos"]


def _draw_schematic(ax, kind, agg_color):
    """Cell = grey disc + nucleus; inclusion = filled; RC = dashed void drawn
    larger than the inclusion, for the margin cut around it; EA = inclusion
    plus a dashed ring for co-excised material."""
    ax.set_xlim(-1.15, 1.15)
    ax.set_ylim(-1.15, 1.15)
    ax.set_aspect("equal")
    ax.axis("off")
    _cell, _edge, _nuc = "0.90", "0.55", "0.78"
    if kind in ("CWO", "CW", "RC"):
        ax.add_patch(mpatches.Circle((0, 0), 1.0, fc=_cell, ec=_edge, lw=AXES_LW))
        ax.add_patch(mpatches.Circle((-0.32, 0.12), 0.42, fc=_nuc, ec=_edge, lw=THIN_LW))
    if kind == "CW":
        ax.add_patch(mpatches.Circle((0.42, -0.38), 0.30, fc=agg_color, ec="black", lw=THIN_LW))
    if kind == "RC":
        ax.add_patch(mpatches.Circle((0.42, -0.38), 0.38, fc="white", ec=_edge,
                                     lw=AXES_LW, ls=(0, (2, 2))))
    if kind == "EA":
        ax.add_patch(mpatches.Circle((0, 0), 0.62, fc=agg_color, ec="black", lw=THIN_LW))
        ax.add_patch(mpatches.Circle((0, 0), 0.88, fc="none", ec=_edge,
                                     lw=AXES_LW, ls=(0, (2, 2))))


# One extra gridspec row of height BOTTOM_PAD is the bottom spacer. The ylim
# below is matched to it so the four dot rows stay centred on the four schematics.
fig = plt.figure(figsize=figsize(2, 1))
gs  = fig.add_gridspec(len(ROW_ORDER) + 1, 3,
                       width_ratios=[0.35, 3.2, 1.0],
                       height_ratios=[1] * len(ROW_ORDER) + [BOTTOM_PAD],
                       wspace=0.06, hspace=0.0)
axL = fig.add_subplot(gs[:, 1])
axR = fig.add_subplot(gs[:, 2], sharey=axL)

# schematic + label per row; grid row 0 is at the top, so reverse ROW_ORDER
for _k, _g in enumerate(reversed(ROW_ORDER)):
    _ax = fig.add_subplot(gs[_k, 0])
    # the inclusion is always drawn in EA's colour, whatever row it appears in
    _draw_schematic(_ax, _g, FRACTION_COLORS["Pure Aggregates"])
    _ax.text(-0.30, 0.5, _g, transform=_ax.transAxes,
             ha="right", va="center", fontsize=TICK_FS, color=_colors[_g],
             fontweight="bold" if _g == "CW" else "normal",
             style="italic" if _g in CONTROLS else "normal")

for ax in (axL, axR):
    ax.axvline(0,   color=REF_COLOR, lw=AXES_LW, zorder=1)
    ax.axvline(100, color=REF_COLOR, lw=AXES_LW, zorder=1)

    for g in ROW_ORDER:
        v, y, c = _pos[g], Y[g], _colors[g]
        _open = g in CONTROLS
        ax.errorbar([v.mean()], [y], xerr=[v.std(ddof=1)], fmt="o",
                    ms=DOT, color=c,
                    mfc="white" if _open else c,
                    mec=c if _open else "black",
                    mew=LINE_LW if _open else MARKER_LW,
                    capsize=CAP,
                    capthick=AXES_LW if _open else LINE_LW,
                    elinewidth=AXES_LW if _open else LINE_LW, zorder=4)
        # x offset compensates for the trailing " %": ha="center" centres the whole
        # string, so without it the digits sit left of the marker.
        # y offset clears the marker radius, so the text does not sit on it.
        ax.annotate(f"{v.mean():.0f} %", (v.mean(), y),
                    xytext=(native_pt(1.2), DOT / 2 + native_pt(1.5)), textcoords="offset points",
                    ha="center", va="bottom", fontsize=ANNOT_FS, color=c,
                    fontweight="bold" if g == "CW" else "normal",
                    bbox=MASK, annotation_clip=True, zorder=5)

for ax, xval, txt in ((axL, 0, "expected 0"), (axR, 100, "expected 100")):
    ax.text(xval, 0.015, txt,
            transform=blended_transform_factory(ax.transData, ax.transAxes),
            ha="center", va="bottom", fontsize=ANNOT_FS, color=REF_COLOR, style="italic",
            bbox=MASK, zorder=5)

# matched to the BOTTOM_PAD spacer row above
axL.set_ylim(-0.5 - BOTTOM_PAD, len(ROW_ORDER) - 0.5)
axL.set_yticks([])
axL.set_xlim(*LEFT_XLIM)
axL.set_xticks(LEFT_TICKS)
axR.set_xlim(*RIGHT_XLIM)
axR.set_xticks(RIGHT_TICKS)

axL.spines["right"].set_visible(False)
axR.spines["left"].set_visible(False)
axR.tick_params(left=False, labelleft=False)

# break marks on the axis line only
_d  = 0.5
_bk = dict(marker=[(-1, -_d), (1, _d)], markersize=BREAK, linestyle="none",
           color="black", mec="black", mew=AXES_LW, clip_on=False)
axL.plot([1], [0], transform=axL.transAxes, **_bk)
axR.plot([0], [0], transform=axR.transAxes, **_bk)

fig.subplots_adjust(**MARGINS)
fig.text(XLABEL_X, XLABEL_Y, "Aggregate content (% of protein mass matching EA)",
         ha="center", fontsize=LABEL_FS)

save_panel(fig, "d", "aggregate_ruler", FIG2_DIR)

## 8.4 Mass-balance test of excision purity (SI2F)

In [ ]:
# ============================================================
# Excision purity: mass-balance models against the measurement noise
# ============================================================
# A cell with aggregates is its excised aggregate plus the rest of the cell:
# CW = w·EA + (1−w)·RC. The weight w is fitted on marker proteins. The statistic
# of a model is the median |log2 residual| over K times the bootstrap SD of the
# residual (the noise floor), so a model whose residual is measurement noise
# scores 1. Each alternative model changes one element: the host fraction, the
# target, a removed component, or the target values shuffled across proteins.
# A fitted w outside 0 to 1 disqualifies a model whatever its residual: a mass
# fraction cannot lie outside that range, and such a w means the target does not
# lie between the two components. The plot draws these models open.
#
# Uses FDR_MAX_RULER, MIN_LOG2FC and MIN_ANCHOR_FC from the ruler cell, so both
# panels select markers by the same rule.

N_BOOT = 200          # bootstrap repeats for the noise floor
NOISE_K = 0.6745      # half of a zero-centred normal falls within 0.6745 SD, so a
                      # correctly specified model gives median|resid| / (K·SD) = 1

PURITY_GROUPS = {
    "CWO": "-BafA1 PFF Cells Without Aggregates",
    "CW":  "-BafA1 PFF Cells With Aggregates",
    "RC":  "-BafA1 PFF Rest Cells",
    "EA":  "-BafA1 PFF Pure Aggregates",
    "PBS": "-BafA1 PBS Cells Without Aggregates",
}

# (label, kind, target, component A, component B, shuffle target)
# A or B None: that component is dropped and w is fixed (1 or 0), not fitted.
# `kind` labels the printed table. "shuffled" permutes the target's values across
# proteins: the distribution stays the same, every value moves to another protein.
# It tests whether the fit depends on which protein has which value.
PURITY_MODELS = [
    ("CW = w·EA + (1−w)·RC",   "under test",        "CW",  "EA",  "RC",  False),
    ("CW = w·EA + (1−w)·CWO",  "host swap",         "CW",  "EA",  "CWO", False),
    ("CW = w·EA + (1−w)·PBS",  "host swap",         "CW",  "EA",  "PBS", False),
    ("CW = EA alone (w=1)",    "component removed", "CW",  "EA",  None,  False),
    ("CW = RC alone (w=0)",    "component removed", "CW",  None,  "RC",  False),
    ("RC = w·EA + (1−w)·CWO",  "target swap",       "RC",  "EA",  "CWO", False),
    ("CWO = w·EA + (1−w)·RC",  "target swap",       "CWO", "EA",  "RC",  False),
    ("CW shuffled = w·EA + (1−w)·RC", "identity shuffled", "CW", "EA", "RC", True),
]

# ------------------------------------------------------------
# 1. The 30 wells (-BafA1: CWO, CW, RC, EA, PBS), complete in all, as compositions
# ------------------------------------------------------------
_in_p    = adata_sf.obs[biological].isin(PURITY_GROUPS.values())
adata_p  = adata_sf[_in_p].copy()
_np_before = adata_p.n_vars
adata_p  = at.pp.filter_data_completeness(adata_p, max_missing=0, action="drop")

_lin_p   = adata_p.to_df()
comp_p   = _lin_p.div(_lin_p.sum(axis=1), axis=0)
labels_p = adata_p.obs[biological]

print(f"Wells:    {adata_p.n_obs}  ("
      + ", ".join(f"{k}={int((labels_p == v).sum())}" for k, v in PURITY_GROUPS.items()) + ")")
print(f"Proteins: {_np_before} -> {adata_p.n_vars} complete in all {adata_p.n_obs} wells")

# ------------------------------------------------------------
# 2. Markers: EA against CWO by the ruler's rule, one set for all models
# ------------------------------------------------------------
_adata_pc = ad.AnnData(
    X=np.log2(comp_p.to_numpy()),
    obs=pd.DataFrame({biological: labels_p.to_numpy()}, index=comp_p.index),
    var=pd.DataFrame(index=comp_p.columns),
)
with contextlib.redirect_stdout(io.StringIO()):
    _sel_p = at.tl.diff_exp_ttest(
        _adata_pc, between_column=biological,
        comparison=(PURITY_GROUPS["EA"], PURITY_GROUPS["CWO"]),
    )
_sel_p = _sel_p[(_sel_p["fdr"] < FDR_MAX_RULER)
                & (_sel_p["log2fc"].abs() >= MIN_LOG2FC)]

_wells_by_group = {g: comp_p.index[(labels_p == lab).to_numpy()]
                   for g, lab in PURITY_GROUPS.items()}

def _means(rng=None):
    """Mean composition per group. With rng, resample each group's 6 wells with
    replacement — this is what generates the noise floor."""
    out = {}
    for g, wells in _wells_by_group.items():
        w = wells if rng is None else rng.choice(wells, size=len(wells), replace=True)
        out[g] = comp_p.loc[w].mean(axis=0)
    return out

_m0 = _means()

# The ruler's arithmetic-anchor guard: the fit divides by (A − B), so markers
# whose arithmetic means are close make the weight unstable. Applied once, on the
# pair of the model under test, so that all models use the same markers.
_pair  = pd.concat([_m0["CWO"][_sel_p.index], _m0["EA"][_sel_p.index]], axis=1)
_ratio_p = _pair.max(axis=1) / _pair.min(axis=1).replace(0, np.nan)
markers_p = _ratio_p.index[(_ratio_p >= MIN_ANCHOR_FC) & _ratio_p.notna()]

print(f"Markers:  {len(_sel_p)} pass FDR<{FDR_MAX_RULER} & |log2FC|>={MIN_LOG2FC}"
      f" -> {len(markers_p)} also clear the {MIN_ANCHOR_FC}x anchor guard")

# ------------------------------------------------------------
# 3. Fit one model
# ------------------------------------------------------------
# The permutation is drawn once and reused in every bootstrap repeat. A new
# permutation per repeat would add the permutation to that model's noise floor
# and hide the failure the model tests.
_PERM = np.random.default_rng(0).permutation(len(comp_p.columns))

def _fit(means, target, A, B, markers, shuffle=False):
    """Returns (weight, residual Series in log2 units) over the marker set."""
    t = means[target]
    if shuffle:
        t = pd.Series(t.to_numpy()[_PERM], index=t.index)
    if B is None:                       # A alone, weight fixed at 1
        w, pred = 1.0, means[A]
    elif A is None:                     # B alone, weight fixed at 0
        w, pred = 0.0, means[B]
    else:
        a, b = means[A], means[B]
        w = float(((t[markers] - b[markers]) / (a[markers] - b[markers])).median())
        pred = w * a + (1 - w) * b
    # An unclipped weight can make a prediction non-positive; its log2 is then
    # undefined, so those proteins are left out of the residual.
    ok = (pred[markers] > 0) & (t[markers] > 0)
    return w, np.log2(t[markers][ok] / pred[markers][ok])

# ------------------------------------------------------------
# 4. Point estimate and bootstrap noise floor
# ------------------------------------------------------------
_rng = np.random.default_rng(1)
_boot = {lbl: [] for lbl, *_ in PURITY_MODELS}
for _ in range(N_BOOT):
    _mb = _means(_rng)
    for lbl, _kind, tgt, A, B, shuf in PURITY_MODELS:
        _boot[lbl].append(_fit(_mb, tgt, A, B, markers_p, shuffle=shuf)[1])

purity_rows = []
for lbl, kind, tgt, A, B, shuf in PURITY_MODELS:
    w, resid = _fit(_m0, tgt, A, B, markers_p, shuffle=shuf)
    med_abs  = float(resid.abs().median())
    noise_sd = float(pd.DataFrame(_boot[lbl]).std(axis=0, ddof=1).median())
    fixed_w  = (A is None or B is None)
    purity_rows.append({
        "label": lbl, "kind": kind, "weight": w, "fixed_w": fixed_w,
        # a fitted mass fraction outside 0..1 is impossible
        "invalid": (not fixed_w) and not (0.0 <= w <= 1.0),
        "median_abs_resid": med_abs, "noise_sd": noise_sd,
        "stat": med_abs / (NOISE_K * noise_sd),
        "n_used": int(resid.notna().sum()),
    })

purity_df = pd.DataFrame(purity_rows).sort_values("stat").reset_index(drop=True)

print(f"\n{'model':<34}{'kind':<20}{'stat':>7}{'w':>8}{'med|r|':>9}{'noiseSD':>9}{'n':>7}")
for _, r in purity_df.iterrows():
    _w = "fixed" if r["fixed_w"] else f"{r['weight']:.2f}"
    _flag = "  <- w outside 0..1, disqualified" if r["invalid"] else ""
    print(f"{r['label']:<34}{r['kind']:<20}{r['stat']:>7.2f}{_w:>8}"
          f"{r['median_abs_resid']:>9.3f}{r['noise_sd']:>9.3f}{r['n_used']:>7}{_flag}")

In [ ]:
# SI2F, 2 x 1 units: the statistic of each model on a log axis; 1 = the residual
# is measurement noise.
# Two tones: the rows are named after fractions, so fraction colours would suggest
# a link to those fractions. The model under test takes the call-out colour,
# every alternative one grey.
COLOR_UNDER_TEST = HIGHLIGHT_COLOR
COLOR_ALT        = "0.45"

NOISE_LABEL_Y = 0.12                  # axes fraction, raise to lift the "= noise" annotation
DOT_TEST, DOT_ALT = native_pt(3.2), native_pt(2.4)    # marker diameters in print

fig, axm = create_figure(1, 1, figsize=figsize(2, 1))
ax = axm.next()

_x_lo = min(0.7, purity_df["stat"].min() * 0.7)
_x_hi = purity_df["stat"].max() * 8

# Dotted guides from the axes edge, next to each row label, to the marker
for _i, r in purity_df.iterrows():
    ax.hlines(_i, _x_lo, r["stat"], color=BG_COLOR, lw=THIN_LW, ls=":", zorder=1)

for _i, r in purity_df.iterrows():
    _under = r["kind"] == "under test"
    _col   = COLOR_UNDER_TEST if _under else COLOR_ALT
    ax.scatter(r["stat"], _i, s=(DOT_TEST if _under else DOT_ALT) ** 2,
               facecolors="none" if r["invalid"] else _col,
               edgecolors=_col if r["invalid"] else "black",
               linewidths=LINE_LW if r["invalid"] else MARKER_LW, zorder=4)
    _w = "w fixed" if r["fixed_w"] else f"w = {r['weight']:.2f}".replace("-", "−")
    # Value and weight in one label right of the point: placed above and below
    # it, they collided with the neighbouring rows.
    ax.annotate(f"{r['stat']:.1f}×   ({_w})", (r["stat"], _i),
                xytext=(native_pt(3.0), 0), textcoords="offset points",
                ha="left", va="center", fontsize=ANNOT_FS,
                fontweight="bold" if _under else "normal",
                color=_col if (_under or r["invalid"]) else "0.25")

ax.axvline(1.0, color=REF_COLOR, lw=AXES_LW, zorder=2)
# y in axes fraction, so the height of the annotation does not depend on the row count
ax.annotate("residual = measurement noise", (1.0, NOISE_LABEL_Y),
            xycoords=blended_transform_factory(ax.transData, ax.transAxes),
            xytext=(native_pt(1.5), 0), textcoords="offset points",
            ha="left", va="center", fontsize=ANNOT_FS, color=REF_COLOR, style="italic")

ax.set_xscale("log")
ax.set_xlim(_x_lo, _x_hi)
# Plain numbers on the log axis: the exponent of a 10^n label prints below 5 pt.
ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
# Unlabelled minor grid lines at 2, 3 and 5 per decade (2, 3, 5, 20, 30, 50, ...):
# most points lie within one decade.
ax.xaxis.set_minor_locator(mpl.ticker.LogLocator(base=10.0, subs=(2.0, 3.0, 5.0),
                                                 numticks=20))
ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())

ax.set_yticks(np.arange(len(purity_df)))
ax.set_yticklabels(purity_df["label"])
for _t, _kind in zip(ax.get_yticklabels(), purity_df["kind"]):
    if _kind == "under test":
        _t.set_fontweight("bold")
        _t.set_color(COLOR_UNDER_TEST)
ax.set_ylim(-0.6, len(purity_df) - 0.4)
ax.invert_yaxis()

ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.set_axisbelow(True)
ax.grid(axis="x", which="major", alpha=0.25, zorder=0)
ax.grid(axis="x", which="minor", alpha=0.12, zorder=0)
ax.grid(axis="y", visible=False)

_handles = [
    Line2D([0], [0], marker="o", ls="none", markerfacecolor=COLOR_UNDER_TEST,
           markeredgecolor="black", markeredgewidth=MARKER_LW, markersize=DOT_TEST,
           label="model under test"),
    Line2D([0], [0], marker="o", ls="none", markerfacecolor=COLOR_ALT,
           markeredgecolor="black", markeredgewidth=MARKER_LW, markersize=DOT_ALT,
           label="alternative model"),
    Line2D([0], [0], marker="o", ls="none", markerfacecolor="none",
           markeredgecolor=COLOR_ALT, markeredgewidth=LINE_LW, markersize=DOT_ALT,
           label="open: fitted w outside 0–1,\ndisqualified"),
]
# Right-aligned: markers on the right, text ending at them. borderaxespad=0 and
# borderpad=0 put the legend in the axes corner, clear of the weight labels.
_leg = ax.legend(handles=_handles, loc="upper right", frameon=False, markerfirst=False,
                 handlelength=1.0, handletextpad=0.5, borderaxespad=0, borderpad=0)
for _t in _leg.get_texts():
    _t.set_multialignment("right")

label_axes(ax, xlabel="Median |residual| ÷ measurement noise")
fig.get_layout_engine().set(h_pad=0.02)

save_panel(fig, "f", "excision_purity_models", SI2_DIR)

# 9. PFF response

## 9.1 Volcano plot, PFF against PBS (Figure 2B)

In [ ]:
# Figure 2B, 1 x 1 unit: volcano plot, PFF against PBS (−BafA1, CWO).
comparison_name = "PFF_vs_PBS_minusBafA1"

always_label = [
    "SNCA",                          # the strongest hit
]

# Thresholds of the colour classes and of the dashed lines
LFC_THRESH = 0.3
FDR_THRESH = 0.05

DOT_BG, DOT_LABEL = native_pt(1.0), native_pt(2.4)     # marker diameters in print
# Leader lines stop short of the label box (A) and outside the labelled dot (B).
SHRINK_A, SHRINK_B = native_pt(0.8), DOT_LABEL / 2 + native_pt(0.4)

result = ttest_results[comparison_name].copy()
result["status"] = "unchanged"
result.loc[(result["fdr"] < FDR_THRESH) & (result["log2fc"] >  LFC_THRESH), "status"] = "upregulated"
result.loc[(result["fdr"] < FDR_THRESH) & (result["log2fc"] < -LFC_THRESH), "status"] = "downregulated"

top_indices = result.index[result["single_gene_name"].isin(always_label)]
result["plot_layer"] = result["status"]
result.loc[result.index.isin(top_indices) & (result["status"] == "upregulated"),   "plot_layer"] = "up_labeled"
result.loc[result.index.isin(top_indices) & (result["status"] == "downregulated"), "plot_layer"] = "down_labeled"
result.loc[result.index.isin(top_indices) & (result["status"] == "unchanged"),     "plot_layer"] = "ns_labeled"

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()
ax.spines[["top", "right"]].set_visible(True)     # boxed: all four axis lines

# Marker settings per layer; volcano() passes the "scatter_kwargs" of a layer to scatter().
_lab = {"scatter_kwargs": {"s": DOT_LABEL ** 2, "alpha": 1.0, "edgecolor": "black", "linewidth": MARKER_LW}}
at.pl.volcano(
    data=result, x_column="log2fc", y_column="-log10(fdr)", ax=ax,
    x_thresholds=(-LFC_THRESH, LFC_THRESH),
    y_thresholds=(-np.log10(FDR_THRESH),),
    layers=[
        ("plot_layer", "up_labeled",    "up",       _lab),
        ("plot_layer", "down_labeled",  "down",     _lab),
        ("plot_layer", "ns_labeled",    "ns_label", _lab),
        ("plot_layer", "upregulated",   "up_bg",    {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": 0.6, "linewidth": 0}}),
        ("plot_layer", "downregulated", "down_bg",  {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": 0.6, "linewidth": 0}}),
        ("plot_layer", "unchanged",     "ns",       {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": 0.03, "linewidth": 0}}),
    ],
    color_dict={
        "up": UP_COLOR, "down": DOWN_COLOR,
        "up_bg": UP_COLOR, "down_bg": DOWN_COLOR,
        "ns_label": "black", "ns": "black",
    },
    label_layers=None,
    display_id_column="single_gene_name",
    legend=None,
)
ax.set_title("")   # the volcano function sets the title "Volcano Plot" and has no parameter for it

# The only lines on the axes so far are the threshold lines, which the volcano
# function draws 1 pt wide on the canvas (0.22 pt in print, below the minimum).
for _line in ax.lines:
    _line.set(linewidth=THIN_LW, color=REF_COLOR)

# x range is needed before the labels, to size their initial nudge
lfc_min, lfc_max = result["log2fc"].min(), result["log2fc"].max()
_dx = 0.03 * (lfc_max - lfc_min)

# Down-regulated labels sit to the left of their dot, up-regulated to the right:
# ha fixes which way the text grows, the nudge starts it clear of the marker.
labeled = result[result.index.isin(top_indices)]
texts = []
for _, row in labeled.iterrows():
    _side = -1 if row["log2fc"] < 0 else 1
    texts.append(
        ax.text(row["log2fc"] + _side * _dx, row["-log10(fdr)"],
                row["single_gene_name"],
                ha="right" if _side < 0 else "left", va="center", fontsize=ANNOT_FS,
                bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                          edgecolor="none", alpha=0.85))
    )

fig.canvas.draw()
# Vertical movement only, so a label can never be dragged back across its dot.
# No arrowprops: adjust_text draws a leader only for labels it moved, so the
# leaders are drawn below.
adjust_text(texts, ax=ax,
            only_move={"text": "y", "static": "y", "explode": "y", "pull": "y"},
            force_text=(0.3, 0.3),
            force_points=(0.2, 0.2),
            expand=(1.2, 1.2))

# One leader line per label. A leader shorter than the two shrink gaps would be
# drawn as a zero-length stub, so it is skipped.
for _t, (_, _row) in zip(texts, labeled.iterrows()):
    _x0, _y0 = _t.get_position()
    _p0 = ax.transData.transform((_x0, _y0))
    _p1 = ax.transData.transform((_row["log2fc"], _row["-log10(fdr)"]))
    _len_pt = float(np.hypot(*(_p1 - _p0))) * 72.0 / fig.dpi
    if _len_pt <= SHRINK_A + SHRINK_B + 1:
        continue
    ax.annotate("", xy=(_row["log2fc"], _row["-log10(fdr)"]), xytext=(_x0, _y0),
                arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW,
                                shrinkA=SHRINK_A, shrinkB=SHRINK_B), zorder=2)

# Labels extend outward from their dots, so they need room at both ends.
x_pad = 0.14 * (lfc_max - lfc_min)
ax.set_xlim(lfc_min - x_pad, lfc_max + x_pad)

# The counts are in the legend, next to the colours.
n_up   = int((result["status"] == "upregulated").sum())
n_down = int((result["status"] == "downregulated").sum())
_handles = [
    Line2D([], [], marker="o", ls="none", ms=DOT_LABEL, color=UP_COLOR,
           mec="black", mew=MARKER_LW, label=f"upregulated ({n_up})"),
    Line2D([], [], marker="o", ls="none", ms=DOT_LABEL, color=DOWN_COLOR,
           mec="black", mew=MARKER_LW, label=f"downregulated ({n_down})"),
]
ax.legend(handles=_handles, loc="lower right", frameon=False,
          title=f"FDR < {FDR_THRESH},  |LFC| > {LFC_THRESH}")

label_axes(ax, xlabel="log2FC (CWO PFF/ CWO PBS)", ylabel="−log10 FDR")

fig.canvas.draw()
_lb = ax.get_legend().get_window_extent()
_hidden = [t.get_text() for t in texts if _lb.overlaps(t.get_window_extent())]
print(f"CHECK labels under the legend (must be none): {_hidden or 'none'}")

save_panel(fig, "b", "volcano_PFF_vs_PBS", FIG2_DIR)

## 9.2 GSEA, PFF against PBS (SI2C)

Gene set enrichment analysis with gseapy prerank on the Welch t-statistic, against Reactome and GO biological process from MSigDB (cached in `src/gsea_genesets_cache.parquet`). The computation is in `src/gsea_pipeline.py`.

- **[1/3]** runs the enrichment.
- **[2/3]** clusters redundant terms by leading-edge overlap and writes the decision file `review.csv`.
- **[3/3]** plots the representative of each cluster as chosen in `review.csv`.

The decisions in `review.csv` (representative term, exclusion and display label of each high-priority cluster) were made by the authors with the help of an AI review (Claude, Anthropic) that followed the rules in `GSEA_REVIEW_PROMPT.md`; they are an input of this analysis. A re-run of [2/3] carries every decision over by term name; a term whose cluster changed is marked MOVED, and [3/3] reports it.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA [1/3]: PFF against PBS (−BafA1, CWO).
# Ranking: Welch t-statistic. Null: gene permutation (prerank).
# ═══════════════════════════════════════════════════════════════════════
gsea_comp       = "PFF_vs_PBS_minusBafA1"
COND_A          = "-BafA1 PFF Cells Without Aggregates"
COND_B          = "-BafA1 PBS Cells Without Aggregates"

# Excluded on lineage or physiology grounds only: each name requires a cell type,
# tissue or organ-level process absent from a HEK293 monolayer, so no gene content
# could make the term mean what it says. Terms whose genes may be relevant here
# stay in and are judged in the decision file on their core genes.
ARTIFACT_PATHWAYS = {
    # Vascular / blood — no endothelium, no vessels, no plasma in the dish
    "GOBP_ENDOTHELIAL_CELL_MIGRATION",
    "GOBP_REGULATION_OF_BLOOD_CIRCULATION",
    "GOBP_REGULATION_OF_SYSTEMIC_ARTERIAL_BLOOD_PRESSURE",
    "GOBP_VASCULAR_PROCESS_IN_CIRCULATORY_SYSTEM",
    "REACTOME_CELL_SURFACE_INTERACTIONS_AT_THE_VASCULAR_WALL",
    "GOCC_BLOOD_MICROPARTICLE",

    # Immune — the named effector cell types are absent
    "GOBP_GRANULOCYTE_MIGRATION",
    "GOBP_POSITIVE_REGULATION_OF_IMMUNE_EFFECTOR_PROCESS",
    "GOBP_T_CELL_MEDIATED_IMMUNITY",

    # Embryonic tissue programmes — cannot occur in a monolayer
    "REACTOME_FORMATION_OF_PARAXIAL_MESODERM",
    "REACTOME_GASTRULATION",
    "REACTOME_SOMITOGENESIS",

    # Melanocytes — absent cell type
    "REACTOME_MITF_M_REGULATED_MELANOCYTE_DEVELOPMENT",
    "REACTOME_TRANSCRIPTIONAL_AND_POST_TRANSLATIONAL_REGULATION_OF_MITF_M_EXPRESSION_AND_ACTIVITY",

    # Cochlear hair cells — absent cell type
    "REACTOME_SENSORY_PROCESSING_OF_SOUND",
}

# Gene permutation only (run_phenotype=False). The phenotype permutation keeps the
# gene-gene correlation, but 6 against 6 samples allow only 462 distinct
# relabellings, so its smallest p-value is about 2.2e-3 and it cannot resolve the
# down direction.
cfg = GSEAConfig(output_root=gsea_dir,
                 artifact_pathways=frozenset(ARTIFACT_PATHWAYS),
                 n_review_clusters=40,
                 run_phenotype=False)

gene_sets = load_gene_sets(cfg)     # cached in src/gsea_genesets_cache.parquet
gsea_res  = run_contrast(adata_to_use, biological, COND_A, COND_B,
                         gsea_comp, gene_sets, cfg)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA [2/3]: cluster redundant terms by leading-edge overlap and write the
# decision file review.csv. Nothing is dropped here.
# Complete linkage: every pair of terms in a cluster must pass the overlap
# threshold; single linkage chained most terms into one cluster with an empty
# core. Decisions already in review.csv are carried over by term name.
# ═══════════════════════════════════════════════════════════════════════
# The decisions are an input. Without the file, build_review would write a new one
# and [3/3] would plot unreviewed clusters under their raw term names.
REVIEW_PATH = cfg.contrast_dir(gsea_comp) / "review.csv"
assert REVIEW_PATH.exists(), f"decision file missing: {REVIEW_PATH}; check gsea_dir"

_ = build_review(gsea_res, cfg)            # the returned table is in review.csv

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA [3/3]: plot the representatives chosen in review.csv. SI2C, 2 x 2 units.
# One row per cluster, TOP_N_PLOT per direction. FDR from gene permutation.
# ═══════════════════════════════════════════════════════════════════════
FDR_THRESH = 0.05
TOP_N_PLOT = 10        # per direction

# Marker area grows linearly with set size, from DOT_MIN to DOT_MAX
DOT_MIN, DOT_MAX = native_pt(4) ** 2, native_pt(9.0) ** 2

# Term names are title-cased for display; these acronyms are restored afterwards.
ACRONYMS = ["RNA", "DNA", "mRNA", "rRNA", "snRNA", "ATP", "NADH",
            "II", "I", "ERAD", "AP", "ER", "COPI", "COPII", "UBI",
            "ABC", "SLC"]

def fix_acronyms(s):
    for a in ACRONYMS:
        s = re.sub(rf'\b{a}\b', a, s, flags=re.IGNORECASE)
    return s

rev = pd.read_csv(REVIEW_PATH, keep_default_na=False)

# Replays the selection from review.csv and stops if a chosen representative
# would be lost.
assert_pass2_selection(rev)

for c in ("NES", "fdr", "cluster_plot_rank", "detected_size", "cluster_size"):
    rev[c] = pd.to_numeric(rev[c], errors="coerce")

# --- Clusters without a decision ---------------------------------------
# NEW: the term is not in the earlier review.csv; MOVED: its cluster changed.
unrev = rev[(rev["review_status"].isin(["NEW", "MOVED"])) &
            (rev["review_priority"] == "HIGH")]
if len(unrev):
    print(f"!! {len(unrev)} HIGH-priority term(s) NEW or MOVED — using auto-suggestion:")
    for _, r in unrev.iterrows():
        print(f"   [{r['review_status']:<5}] {r['term'][:70]}")
    print()
report_undecided(rev)

# --- Apply exclusions --------------------------------------------------
choice   = rev["display_choice"].astype(str).str.strip().str.lower()
excluded = set(rev.loc[choice == "exclude", "cluster_id"])
if excluded:
    ex_names = (rev[rev["cluster_id"].isin(excluded) & rev["is_suggested_rep"]
                    .astype(str).str.lower().isin(["true", "1"])]["term"].tolist())
    print(f"Excluded {len(excluded)} cluster(s):")
    for n in ex_names:
        print(f"   {n[:74]}")
rev = rev[~rev["cluster_id"].isin(excluded)]

# --- Representatives ---------------------------------------------------
reps = rev[rev["display_choice"].astype(str).str.strip().str.lower() == "rep"].copy()
dupes = reps.groupby("cluster_id").size()
for cid, n in dupes[dupes != 1].items():
    print(f"!! cluster {cid} has {n} rows marked 'rep' — keeping the strongest |NES|")
reps["_abs"] = reps["NES"].abs()
reps = reps.sort_values("_abs", ascending=False).drop_duplicates("cluster_id", keep="first")

missing = set(rev["cluster_id"]) - set(reps["cluster_id"])
if missing:
    print(f"!! {len(missing)} cluster(s) have NO 'rep' and are omitted")

# --- Labels ------------------------------------------------------------
def _clean(s):
    for p in ["REACTOME_", "GOBP_", "GOCC_", "GOMF_"]:
        s = s.replace(p, "")
    return fix_acronyms(s.replace("_", " ").title())

def _base_label(r):
    """Display label without the count of collapsed terms."""
    return str(r["display_label"]).strip() or _clean(r["term"])

def _label(r):
    base = _base_label(r)
    n = int(r["cluster_size"])
    return f"{base} ({n})" if n > 1 else base

reps["label_base"] = reps.apply(_base_label, axis=1)
reps["label"]      = reps.apply(_label, axis=1)

sel = pd.concat([
    reps[reps["NES"] > 0].nlargest(TOP_N_PLOT, "NES"),
    reps[reps["NES"] < 0].nsmallest(TOP_N_PLOT, "NES"),
]).sort_values("NES").reset_index(drop=True)

n_up, n_dn = int((sel["NES"] > 0).sum()), int((sel["NES"] < 0).sum())
print(f"\nPlotting {len(sel)} clusters ({n_up} up, {n_dn} down)")
if n_up < TOP_N_PLOT or n_dn < TOP_N_PLOT:
    print(f"   NOTE: fewer than {TOP_N_PLOT} available in a direction after exclusions — "
          f"raise n_review_clusters in the config if you want more headroom.")

# --- Plot: horizontal NES dot plot -------------------------------------
# NES is signed, a direction of change, so the colour map is DIRECTION_CMAP.
vmax = np.ceil(sel["NES"].abs().max())
norm = plt.Normalize(vmin=-vmax, vmax=vmax)
smin, smax = sel["detected_size"].min(), sel["detected_size"].max()

def _area(n):
    return DOT_MIN + (n - smin) / max(smax - smin, 1) * (DOT_MAX - DOT_MIN)

fig, axm = create_figure(1, 1, figsize=figsize(2, 2))
ax = axm.next()
ax.set_axisbelow(True)
ax.grid(axis="y", visible=False)

for i, r in sel.iterrows():
    if r["fdr"] >= FDR_THRESH:
        ax.scatter(r["NES"], i, s=_area(r["detected_size"]), facecolors="none",
                   edgecolors="0.55", linewidths=LINE_LW, zorder=3)
    else:
        ax.scatter(r["NES"], i, s=_area(r["detected_size"]), c=[r["NES"]],
                   cmap=DIRECTION_CMAP, norm=norm, edgecolors="black",
                   linewidths=MARKER_LW, zorder=3)

ax.set_yticks(range(len(sel)))
# Arial has no superscript plus (U+207A), which matplotlib draws as an empty box.
ax.set_yticklabels(sel["label"].str.replace("⁺", "+"))
ax.axvline(0, color=REF_COLOR, lw=AXES_LW, zorder=1)
ax.set_xlim(-vmax, vmax)
ax.set_ylim(-0.7, len(sel) - 0.3)
label_axes(ax, xlabel="NES")

sm = plt.cm.ScalarMappable(cmap=DIRECTION_CMAP, norm=norm)
cb = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.02)
cb.set_label("NES")

# Rows are sorted by NES: the down rows (bottom) have their dots on the left, the
# up rows (top) on the right. The upper left and the lower right are empty, and
# the two keys go there.
legend_sizes = np.linspace(smin, smax, 3).astype(int)
size_handles = [
    Line2D([0], [0], marker="o", ls="none", markerfacecolor="0.6",
           markeredgecolor="black", markeredgewidth=MARKER_LW,
           markersize=np.sqrt(_area(s)), label=str(s))
    for s in legend_sizes
]
size_legend = ax.legend(handles=size_handles, title="Set size", loc="upper left",
                        frameon=False)
ax.add_artist(size_legend)

# Two states only: the FDR is from gene permutation, and there is no second null.
sig_handles = [
    Line2D([0], [0], marker="o", ls="none", markerfacecolor="0.75",
           markeredgecolor="black", markeredgewidth=MARKER_LW, markersize=native_pt(2.4),
           label=f"FDR < {FDR_THRESH}"),
    Line2D([0], [0], marker="o", ls="none", markerfacecolor="none",
           markeredgecolor="0.55", markeredgewidth=LINE_LW, markersize=native_pt(2.4),
           label=f"FDR ≥ {FDR_THRESH}"),
]
ax.legend(handles=sig_handles, title="Significance", loc="lower right", frameon=False)

save_panel(fig, "c", "GSEA_PFF_vs_PBS", SI2_DIR)

# 10. Linearity across fractions

## 10.1 ANOVA, post-hoc t-tests and classification

Per BafA1 state: one-way ANOVA across CWO, RC, CW and EA (PFF); post-hoc Welch t-tests of RC against CWO and of EA against CW on the ANOVA-significant proteins; each of these proteins is classified by the directions of the two post-hoc results.

In [ ]:
def diff_exp_anova(
    adata,
    between_column,
    groups=None,
    min_valid_values=2,
):
    """One-way ANOVA per protein across several groups (scipy f_oneway), with a
    Benjamini-Hochberg FDR over the tested proteins. alphapepttools 0.2.0 has no
    ANOVA; the output has the layout of its diff_exp_ttest tables.

    Parameters
    ----------
    adata : ad.AnnData
        AnnData object (log2-transformed).
    between_column : str
        Column in adata.obs defining groups.
    groups : list of str, optional
        Subset of groups to test. If None, uses all groups in between_column.
    min_valid_values : int
        Minimum non-NaN values required per group per protein. A protein below it
        in any group is not tested (NaN).

    Returns
    -------
    pd.DataFrame
        Columns: protein, f_statistic, p_value, -log10(p_value), fdr, -log10(fdr),
        method, groups.
    """
    if between_column not in adata.obs.columns:
        raise ValueError(f"Column '{between_column}' not found in adata.obs.")

    if groups is None:
        groups = adata.obs[between_column].dropna().unique().tolist()

    # Per-group DataFrames
    group_dfs = {}
    for g in groups:
        mask = adata.obs[between_column] == g
        if mask.sum() < min_valid_values:
            raise ValueError(f"Group '{g}' has only {mask.sum()} samples, need at least {min_valid_values}.")
        group_dfs[g] = adata[mask].to_df()

    print(f"ANOVA across {len(groups)} groups: {groups}")
    print(f"  Samples per group: {', '.join(f'{g}={len(df)}' for g, df in group_dfs.items())}")

    proteins = adata.var_names
    f_stats = []
    p_values = []

    for protein in proteins:
        arrays = []
        skip = False
        for g in groups:
            vals = group_dfs[g][protein].dropna()
            if len(vals) < min_valid_values:
                skip = True
                break
            arrays.append(vals.values)

        if skip or len(arrays) < 2:
            f_stats.append(np.nan)
            p_values.append(np.nan)
        else:
            f, p = f_oneway(*arrays)
            f_stats.append(f)
            p_values.append(p)

    # Benjamini-Hochberg over the tested proteins
    p_arr = np.array(p_values)
    fdr = np.full_like(p_arr, np.nan, dtype=np.float64)
    valid = ~np.isnan(p_arr)
    if valid.any():
        _, fdr_valid, _, _ = multipletests(p_arr[valid], method="fdr_bh")
        fdr[valid] = fdr_valid

    def neg_log10(x):
        if np.isnan(x):
            return np.nan
        return 300.0 if x == 0 else -np.log10(x)     # p = 0 (below double precision) -> 300

    result = pd.DataFrame({
        "protein": proteins,
        "f_statistic": f_stats,
        "p_value": p_values,
        "-log10(p_value)": [neg_log10(p) for p in p_values],
        "fdr": fdr,
        "-log10(fdr)": [neg_log10(f) for f in fdr],
        "method": "anova",
        "groups": ", ".join(groups),
    })

    n_sig = (result["fdr"] < 0.05).sum()
    print(f"  Tested: {valid.sum()} proteins, significant (FDR<0.05): {n_sig}")

    return result

In [ ]:
# PFF fractions in order of measured aggregate content (Figure 2D), the ANOVA
# groups per BafA1 state, and the two post-hoc pairs.
order = [
    "PFF Cells Without Aggregates",
    "PFF Rest Cells",
    "PFF Cells With Aggregates",
    "PFF Pure Aggregates",
]

anova_comparisons = {
    "ANOVA_PFF_minusBafA1": [f"-BafA1 {g}" for g in order],
    "ANOVA_PFF_plusBafA1": [f"+BafA1 {g}" for g in order],
}

# Post-hoc pairwise comparisons per BafA1 condition
posthoc_pairs = {
    "RestCells_vs_CellsWithout": ("PFF Rest Cells", "PFF Cells Without Aggregates"),
    "PureAggs_vs_CellsWith": ("PFF Pure Aggregates", "PFF Cells With Aggregates"),
}

In [ ]:
# ANOVA per BafA1 state (PFF fractions only), then the post-hoc t-tests on the
# ANOVA-significant proteins; their FDR is computed over these proteins only.

anova_results = {}
for name, groups in anova_comparisons.items():
    result = diff_exp_anova(
        adata=adata_to_use,
        between_column=biological,
        groups=groups,
    )
    if result is not None:
        anova_results[name] = result
        n_sig = (result["fdr"] < 0.05).sum()
        print(f"{name}: {len(result)} tested, {n_sig} sig (FDR<0.05)")
        result.to_csv(f"{output_dir}/{today}_anova_{filter_label}_{name}.csv", index=False)
    else:
        print(f"{name}: FAILED")

# Post-hoc t-tests (only for ANOVA-significant proteins)
print(f"\n{'='*60}")
print("Post-hoc t-tests (on ANOVA-significant proteins)")
print(f"{'='*60}")

posthoc_results = {}
for anova_name, groups in anova_comparisons.items():
    baf_prefix = "-BafA1" if "minus" in anova_name else "+BafA1"
    baf_label = "minusBafA1" if "minus" in anova_name else "plusBafA1"
    sig_proteins = anova_results[anova_name][anova_results[anova_name]["fdr"] < 0.05]["protein"].values

    # Subset adata to ANOVA-significant proteins only
    adata_sig = adata_to_use[:, adata_to_use.var_names.isin(sig_proteins)].copy()
    print(f"\n{anova_name}: {len(sig_proteins)} ANOVA-significant proteins")

    for pair_name, (g1_base, g2_base) in posthoc_pairs.items():
        full_name = f"posthoc_{pair_name}_{baf_label}"
        g1 = f"{baf_prefix} {g1_base}"
        g2 = f"{baf_prefix} {g2_base}"

        result = at.tl.diff_exp_ttest(
            adata=adata_sig,
            between_column=biological,
            comparison=(g1, g2),
        )
        if result is not None:
            posthoc_results[full_name] = result
            n_sig = (result["fdr"] < 0.05).sum()
            print(f"  {full_name}: {len(result)} tested, {n_sig} sig (FDR<0.05)")
            result.to_csv(f"{output_dir}/{today}_{full_name}.csv", index=False)

print(f"\nCSVs saved to {output_dir}")

In [ ]:
# Classification of the ANOVA-significant proteins by the two post-hoc tests:
#   RC vs CWO   change in the rest of the cell, against cells without aggregates
#   EA vs CW    enrichment in the aggregate, against the whole cell
# Each test gives "↑ sig", "↓ sig" or "n.s." (FDR < fdr_thresh and |log2FC| >
# lfc_thresh); the pair of results gives the category in `keywords`.
# pa_ stands for EA (obs value "Pure Aggregates").
fdr_thresh = 0.05
lfc_thresh = 0.3 # minimum |log2FC| to count as a real signal

def classify_protein(rc_lfc, rc_fdr, pa_lfc, pa_fdr, fdr_thresh=fdr_thresh, lfc_thresh=lfc_thresh):
    rc = "↑ sig" if (rc_fdr < fdr_thresh and rc_lfc > lfc_thresh) else \
         "↓ sig" if (rc_fdr < fdr_thresh and rc_lfc < -lfc_thresh) else "n.s."
    pa = "↑ sig" if (pa_fdr < fdr_thresh and pa_lfc > lfc_thresh) else \
         "↓ sig" if (pa_fdr < fdr_thresh and pa_lfc < -lfc_thresh) else "n.s."

    keywords = {
        ("↑ sig", "↑ sig"): "Sequestration + compensation",
        ("↓ sig", "↑ sig"): "Sequestration only",
        ("↑ sig", "↓ sig"): "Compensation only",
        ("↓ sig", "↓ sig"): "Depleted",
        ("↑ sig", "n.s."): "Cellular upregulation",
        ("↓ sig", "n.s."): "Cellular depletion",
        ("n.s.", "↑ sig"): "Aggregate enriched",
        ("n.s.", "↓ sig"): "Aggregate excluded",
        ("n.s.", "n.s."): "No pattern",
    }
    return rc, pa, keywords[(rc, pa)]

print(f"Thresholds: FDR < {fdr_thresh}, |log2FC| > {lfc_thresh}")

classification_tables = {}
for anova_name, groups in anova_comparisons.items():
    baf_label = "minusBafA1" if "minus" in anova_name else "plusBafA1"
    baf_display = "−BafA1" if "minus" in anova_name else "+BafA1"

    anova_res = anova_results[anova_name]
    rc_res = posthoc_results[f"posthoc_RestCells_vs_CellsWithout_{baf_label}"]
    pa_res = posthoc_results[f"posthoc_PureAggs_vs_CellsWith_{baf_label}"]

    # ANOVA FDR and both post-hoc results per protein
    merged = anova_res[["protein", "fdr"]].rename(columns={"fdr": "anova_fdr"})
    merged = merged.merge(
        rc_res[["protein", "log2fc", "fdr"]].rename(columns={"log2fc": "rc_lfc", "fdr": "rc_fdr"}),
        on="protein", how="left",
    )
    merged = merged.merge(
        pa_res[["protein", "log2fc", "fdr"]].rename(columns={"log2fc": "pa_lfc", "fdr": "pa_fdr"}),
        on="protein", how="left",
    )

    # Add gene names
    name_map = adata_to_use.var["single_gene_name"].to_dict()
    merged["gene"] = merged["protein"].map(name_map)

    # Classify only ANOVA-significant proteins
    rows = []
    for _, row in merged.iterrows():
        anova_sig = row["anova_fdr"] < 0.05
        if anova_sig:
            rc_dir, pa_dir, keyword = classify_protein(
                row["rc_lfc"], row["rc_fdr"], row["pa_lfc"], row["pa_fdr"],
                fdr_thresh=fdr_thresh, lfc_thresh=lfc_thresh,
            )
        else:
            rc_dir, pa_dir, keyword = "–", "–", "Not ANOVA significant"
        rows.append({
            "protein": row["protein"],
            "gene": row["gene"],
            "anova_fdr": row["anova_fdr"],
            "anova_sig": anova_sig,
            "RC vs CWO (LFC)": row["rc_lfc"],
            "RC vs CWO (FDR)": row["rc_fdr"],
            "RC vs CWO": rc_dir,
            "PA vs CW (LFC)": row["pa_lfc"],
            "PA vs CW (FDR)": row["pa_fdr"],
            "PA vs CW": pa_dir,
            "keyword": keyword,
        })

    class_df = pd.DataFrame(rows)
    classification_tables[baf_label] = class_df
    class_df.to_csv(f"{output_dir}/{today}_anova_classification_{baf_label}.csv", index=False)

    # Summary
    sig_df = class_df[class_df["anova_sig"]]
    print(f"\n{'='*60}")
    print(f"Classification summary ({baf_display})")
    print(f"  ANOVA significant: {len(sig_df)} / {len(class_df)}")
    print(f"{'='*60}")
    for keyword, count in sig_df["keyword"].value_counts().items():
        print(f"  {keyword:40s} {count:4d}")

## 10.2 Trajectory plots (Figure 2E; the +BafA1 panel is not yet placed in a figure)

In [ ]:
# Mean log2 intensity of each protein in CWO, RC, CW and EA relative to CWO, per
# classification category: one grey line per protein, the category mean in black,
# three named proteins in the call-out colour. Four subplots in a 1 x 3 unit
# panel. −BafA1: Figure 2E; +BafA1: saved into SI3_DIR with the placeholder
# letter "x".
categories_to_plot = [
    "Sequestration + compensation",
    "Sequestration only",
    "Compensation only",
    "Depleted",
]

# Display names; the keys are the category names in the classification tables.
CATEGORY_DISPLAY = {
    "Sequestration + compensation": "Sequestration +\ncompensation",
    "Sequestration only":           "Sequestration only",
    "Compensation only":            "Compensation only",
    "Depleted":                     "Depletion",
}

# The x order CWO -> RC -> CW -> EA is the order of measured aggregate content
# (Figure 2D).
XLABEL = "Fraction (increasing\naggregate content)"   # on one line it is wider than the panel
YLABEL = "Δlog2 intensity\n(vs CWO)"

# create_figure() uses constrained_layout, so tight_layout() and subplots_adjust()
# are ignored. The spacing goes through the layout engine: hspace is a fraction of
# the subplot height, h_pad a padding in inches. Lower = tighter.
HSPACE, H_PAD = 0.06, 0.02

condition_order = [
    "PFF Cells Without Aggregates",
    "PFF Rest Cells",
    "PFF Cells With Aggregates",
    "PFF Pure Aggregates",          # obs value; displayed as EA
]
condition_labels = ["CWO", "RC", "CW", "EA"]

# protein ID -> gene symbol, and back for the named genes
name_map     = adata_to_use.var["single_gene_name"].to_dict()
gene_to_prot = {gene: pid for pid, gene in name_map.items()}


def _ramp(color, n=3, lightest=0.55):
    """n shades of one colour, light -> full, by blending toward white."""
    base = np.array(mcolors.to_rgb(color))
    return [tuple(base + (1 - base) * f) for f in np.linspace(lightest, 0, n)]

# The three named proteins of a category: shades of the call-out colour.
HIGHLIGHT_RAMP = _ramp(HIGHLIGHT_COLOR, 3)

# Panel letter and folder per BafA1 state. "x" is a placeholder until the +BafA1
# panel has a place in a figure.
SAVE_TO = {"minusBafA1": ("e", FIG2_DIR),
           "plusBafA1":  ("x", SI3_DIR)}

# The proteins named in each category, chosen by hand.
labels_per_category = {
    "minusBafA1": {
        "Sequestration + compensation": ["PCK2", "SQLE"],
        "Sequestration only":           ["GOLGB1", "FUT8", "MAN1B1"],
        "Compensation only":            ["APOE", "PLK1", "JUN"],
        "Depleted":                     ["H2AC11", "HMGN5", "TP53BP1"],

    },
    "plusBafA1": {
        "Sequestration + compensation": ["IFRD1", "MT-CYB", "LAMP2"],
        "Sequestration only":           ["SQSTM1", "FTH1", "NCOA4"],
        "Compensation only":            ["HNRNPC", "CDK1", "POLR1A"],
        "Depleted":                     ["HMGA1", "PAWR", "DAG1"],
    },
}


def _render_category(ax, cat, class_df, intensity_matrix, fig, highlight_genes,
                     show_xlabel=True, show_legend=False):
    """Draw one category into `ax` and return its proteins and named genes.
    highlight_genes are the gene symbols drawn in the call-out colour and
    labelled. show_xlabel switches the x label and the x tick labels (bottom
    subplot only); show_legend puts the "Category mean" key in this subplot."""
    proteins = class_df[class_df["keyword"] == cat]["protein"].values
    cat_display = CATEGORY_DISPLAY.get(cat, cat)

    if len(proteins) == 0:
        ax.set_title(f"{cat_display}\n(no proteins)", fontweight="bold")
        return {"category": cat, "all_genes": [], "highlighted_genes": []}

    trajectories = intensity_matrix.loc[proteins].copy()
    trajectories = trajectories.subtract(trajectories.iloc[:, 0], axis=0)

    for prot in trajectories.index:
        ax.plot(range(4), trajectories.loc[prot].values,
                color=BG_COLOR, alpha=0.45, linewidth=THIN_LW)
    _mean = trajectories.mean(axis=0).values
    ax.plot(range(4), _mean, color="black", linewidth=LINE_LW, zorder=5,
            label="Category mean")

    resolved = [(g, gene_to_prot.get(g)) for g in highlight_genes]
    present  = [(g, p) for g, p in resolved if p in trajectories.index]
    missing  = [g for g, p in resolved if p not in trajectories.index]
    if missing:
        print(f"  [warn] {cat}: gene(s) not in trajectory set: {missing}")
    highlighted = pd.Index([p for _, p in present])

    texts = []
    for prot, color in zip(highlighted, HIGHLIGHT_RAMP):
        vals = trajectories.loc[prot].values
        ax.plot(range(4), vals, color=color, alpha=0.9, linewidth=AXES_LW, zorder=4)
        gene = str(name_map.get(prot, prot))
        texts.append(ax.text(3, vals[-1], gene, fontsize=ANNOT_FS, color="black",
                             ha="center", va="center", zorder=6,
                             bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                                       edgecolor="none", alpha=0.85)))

    ax.margins(y=0.15)
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    corner_x = np.array([xlim[0], xlim[0], xlim[1], xlim[1]])
    corner_y = np.array([ylim[0], ylim[1], ylim[0], ylim[1]])
    obs_x = np.concatenate([np.tile(np.arange(4), len(trajectories)), corner_x])
    obs_y = np.concatenate([trajectories.values.flatten(), corner_y])

    fig.canvas.draw()
    adjust_text(
        texts, ax=ax, x=obs_x, y=obs_y,
        only_move={"text": "xy", "static": "xy", "explode": "xy", "pull": "y"},
        expand=(1.5, 3.0),
        force_text=(0.4, 1.5),
        force_static=(0.8, 1.2),
        arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW),
    )

    ax.set_xticks(range(4))
    ax.set_xticklabels(condition_labels)
    ax.set_ylabel(YLABEL)
    if show_xlabel:
        ax.set_xlabel(XLABEL)
    else:
        # identical categories on every subplot; only the bottom one needs them
        ax.tick_params(axis="x", labelbottom=False)
    ax.axhline(0, color=REF_COLOR, linestyle="--", linewidth=THIN_LW)
    # Spines above the white label boxes: adjust_text can push a label onto the
    # axis line, and the box would hide that part of it.
    for _spine in ax.spines.values():
        _spine.set_zorder(7)
    ax.spines[["top", "right"]].set_visible(True)  # boxed: all four axis lines
    # n goes in the title: inside the axes it collided with the legend and labels
    ax.set_title(f"{cat_display} (n = {len(proteins)})", fontweight="bold")
    if show_legend:
        ax.legend(loc="best", frameon=False)
    return {
        "category": cat,
        "all_genes":  sorted(str(name_map.get(p, p)) for p in proteins),
        "highlighted_genes": [str(name_map.get(p, p)) for p in highlighted],
    }


for baf_label, baf_prefix, baf_display in [
    ("minusBafA1", "-BafA1", "−BafA1"),
    ("plusBafA1", "+BafA1", "+BafA1"),
]:
    class_df = classification_tables[baf_label]
    conditions = [f"{baf_prefix} {c}" for c in condition_order]

    intensity_dict = {}
    for cond in conditions:
        mask = adata_to_use.obs[biological] == cond
        sub = adata_to_use[mask].to_df()
        intensity_dict[cond] = sub.mean(axis=0)
    intensity_matrix = pd.DataFrame(intensity_dict)

    print(f"\n{'='*60}")
    print(f"Protein trajectories ({baf_display})")
    print(f"{'='*60}")

    # x label and x tick labels on the bottom subplot only
    fig_v, axm_v = create_figure(4, 1, figsize=figsize(1, 3))
    category_info = []
    _last = len(categories_to_plot) - 1
    for _i, cat in enumerate(categories_to_plot):
        highlight = labels_per_category[baf_label][cat]
        info = _render_category(axm_v.next(), cat, class_df, intensity_matrix, fig_v,
                                highlight_genes=highlight,
                                show_xlabel=(_i == _last),
                                show_legend=(_i == 0))
        category_info.append(info)

    # Protein list per category
    for info in category_info:
        cat = info["category"]
        all_genes = info["all_genes"]
        named = info["highlighted_genes"]
        if not all_genes:
            print(f"\n--- {cat} (n=0) ---")
            continue
        print(f"\n--- {cat} (n={len(all_genes)}) ---")
        print(f"  highlighted: {', '.join(named)}")
        # Wrap the gene list at 90 characters
        line = "  all genes: "
        for g in all_genes:
            if len(line) + len(g) + 2 > 90:
                print(line)
                line = "             "
            line += g + ", "
        if line.strip():
            print(line.rstrip(", "))

    fig_v.get_layout_engine().set(hspace=HSPACE, h_pad=H_PAD)

    # adjust_text keeps a label inside the axes, but can leave it touching an axis
    # line (MAN1B1, −BafA1). Widen the y range of each subplot until every label
    # box is 1 pt (printed) inside it; labels and leader lines keep their data
    # positions. Checked after the layout is set, which fixes the subplot heights.
    _pad = native_pt(1.0) * fig_v.dpi / 72
    for _ax in fig_v.axes:
        _label_texts = [t for t in _ax.texts if t.get_bbox_patch() is not None]
        for _ in range(20):
            fig_v.canvas.draw()
            _inner = _ax.get_window_extent()
            _boxes = [t.get_bbox_patch().get_window_extent() for t in _label_texts]
            _top    = any(b.y1 > _inner.y1 - _pad for b in _boxes)
            _bottom = any(b.y0 < _inner.y0 + _pad for b in _boxes)
            if not (_top or _bottom):
                break
            _y0, _y1 = _ax.get_ylim()
            _step = 0.03 * (_y1 - _y0)
            _ax.set_ylim(_y0 - _step * _bottom, _y1 + _step * _top)

    _letter, _folder = SAVE_TO[baf_label]
    save_panel(fig_v, _letter, f"linearity_trajectories_{baf_label}", _folder)

# 11. BafA1 effect

## 11.1 BafA1 effect dot plot (Figure 2G)

In [ ]:
# BafA1 hits, selected without a gene list: proteins significant in at least
# MIN_SIG_COMPS of the five BafA1-effect t-tests, with the same sign in every test
# that has a value, and a largest |log2FC| of at least MIN_MAX_ABS_LFC.

# --- Selection criteria ---
SIG_THRESHOLD   = 0.05   # FDR threshold for "significant"
MIN_SIG_COMPS   = 2      # protein must be significant in >= this many of the 5 comparisons
MIN_MAX_ABS_LFC = 0.65   # max|log2FC| across the 5 comparisons must reach this
# --------------------------

bafa1_comparisons = [
    "BafA1_effect_PBS",
    "BafA1_effect_PFF_CellsWithout",
    "BafA1_effect_PFF_CellsWith",
    "BafA1_effect_PFF_PureAggs",
    "BafA1_effect_PFF_RestCells",
]

# Per-protein log2fc and FDR matrices across the 5 comparisons
all_proteins = set()
for comp in bafa1_comparisons:
    all_proteins.update(ttest_results[comp]["protein"])

lfc_mat = pd.DataFrame(index=sorted(all_proteins), columns=bafa1_comparisons, dtype=float)
fdr_mat = pd.DataFrame(index=sorted(all_proteins), columns=bafa1_comparisons, dtype=float)
for comp in bafa1_comparisons:
    res = ttest_results[comp].set_index("protein")
    lfc_mat[comp] = res["log2fc"]
    fdr_mat[comp] = res["fdr"]

# Step 1: significant (FDR < SIG_THRESHOLD) in >= MIN_SIG_COMPS comparisons
n_sig = (fdr_mat < SIG_THRESHOLD).sum(axis=1)
candidates = lfc_mat.index[n_sig >= MIN_SIG_COMPS]
print(f"Significant in >= {MIN_SIG_COMPS} of {len(bafa1_comparisons)} comparisons: {len(candidates)} proteins")

# Step 2: directional consistency — all non-NaN log2fcs share sign
def _consistent(lfc_row):
    signs = np.sign(lfc_row.dropna().values)
    nonzero = signs[signs != 0]
    return len(nonzero) > 0 and len(np.unique(nonzero)) == 1

dir_mask = lfc_mat.loc[candidates].apply(_consistent, axis=1)
candidates = candidates[dir_mask]
print(f"After directional consistency filter:        {len(candidates)} proteins")

# Step 3: max |log2FC| across the 5 comparisons >= MIN_MAX_ABS_LFC
max_abs_lfc = lfc_mat.loc[candidates].abs().max(axis=1)
candidates = candidates[max_abs_lfc >= MIN_MAX_ABS_LFC]
print(f"After max|log2FC| >= {MIN_MAX_ABS_LFC} filter:            {len(candidates)} proteins")

# --- Summary table ---
gene_map = adata_to_use.var["single_gene_name"].to_dict()

selected_df = pd.DataFrame({
    "protein": list(candidates),
    "gene":    [gene_map.get(p, p) for p in candidates],
})
for comp in bafa1_comparisons:
    short = comp.replace("BafA1_effect_", "")
    selected_df[f"log2fc_{short}"] = selected_df["protein"].map(lfc_mat[comp])

lfc_cols = [c for c in selected_df.columns if c.startswith("log2fc_")]

def _direction(row):
    vals = [v for v in row[lfc_cols] if not pd.isna(v)]
    if not vals: return "none"
    if all(v > 0 for v in vals): return "up"
    if all(v < 0 for v in vals): return "down"
    return "mixed"

selected_df["direction"]      = selected_df.apply(_direction, axis=1)
selected_df["max_abs_log2fc"] = selected_df[lfc_cols].abs().max(axis=1)

selected_df["n_sig"]          = selected_df["protein"].map(n_sig)
selected_df = selected_df.sort_values(
    ["direction", "max_abs_log2fc"], ascending=[True, False]
).reset_index(drop=True)

print(f"\nDirection split: {selected_df['direction'].value_counts().to_dict()}")
print("\nFinal selection:")
print(selected_df[["gene", "direction", "n_sig", "max_abs_log2fc"]].to_string(index=False))
selected_df.to_csv(f"{output_dir}/{today}_bafa1_unbiased_hits.csv", index=False)

In [ ]:
# Figure 2G, 1 x 3 units: BafA1 effect (+BafA1 against −BafA1) in each fraction,
# for the hits of the previous cell. ttest_dotplot (src/dotplot.py) also draws
# Figure 2F, SI3B and SI3E. gene_map comes from the previous cell.
bafa1_labels = ["PBS CWO", "CWO", "CW", "EA", "RC"]      # x tick labels, rotated 90°
bafa1_cbar_label = "log2 FC (+BafA1 / −BafA1)"

# The hits of the previous cell, grouped by function. IREB2, NCOA4 and FTL are
# hits but are left out here because SI3E (ferritinophagy) shows them. SQSTM1,
# TAX1BP1 and MAP1LC3B are added by hand, and so are the negative controls.
fig2g_groups = [
    ("TM receptors", [
        "ITM2B", "ITM2C", "APP", "APLP2",                         # APP / ITM families
        "FAT1", "LDLR", "FGFR3", "CD320",
        "TNFRSF10B", "NOTCH2", "EPHB4", "EPHA2", "EPHA7",         # Eph receptors
        "ROBO1", "NEO1",                                          # axon guidance
        "FZD2",                                                   # Wnt
        "IFNGR1", "HLA-A",                                        # immune
        "NECTIN2", "MPZL1", "ANTXR1", "ANTXR2",                   # adhesion / toxin receptors
        "PTPRS", "SGCE",
    ]),
    ("TM transporters", [
        "SLC4A2",    # bicarbonate
        "SLC2A1",    # GLUT1, glucose
        "SLC7A2",    # cationic amino acid
        "SLC20A1",   # phosphate
        "SLC39A14",  # ZIP14, zinc/manganese
    ]),
    ("Secreted", [
        "ADAMTS1",   # protease
        "CCN1",      # matricellular
        "SLIT2",     # axon guidance
        "EMILIN2",   # ECM
        "IGFBPL1",   # IGFBP-like
        "GAL",       # galanin neuropeptide
    ]),
    ("Golgi / ER", [
        "TGOLN2",    # TGN
        "GOLIM4",    # Golgi membrane (↓)
        "KCT2",      # Golgi/ER TM
        "TMEM59",    # ER → LC3-ATG16L1 recruitment
        "RNF149",    # ER ubiquitin ligase
    ]),
    ("Autophagy", [
        "CALCOCO1",
        "CALCOCO2",  # NDP52
        # --- Added by hand: canonical aggrephagy receptors and LC3B, below the selection thresholds ---
        "SQSTM1",    # p62        (manual)
        "TAX1BP1",   #           (manual)
        "MAP1LC3B",  # LC3B       (manual)
    ]),
    ("Other", [
        "ADAM10",    # surface protease
        "JUN",       # AP-1 TF
        "HS6ST2",    # heparan sulfate sulfotransferase
        "PPARD",     # nuclear receptor (↓)
    ]),
    ("Negative controls", [
        # Added by hand: housekeeping and proteasome-core proteins not expected to change
        "ACTB", "GAPDH", "TUBB", "H4C1", "PSMA1",
    ]),
]

ttest_dotplot(fig2g_groups, ttest_results, gene_map, bafa1_comparisons, bafa1_labels,
              "g", "BafA1_effect_dotplot", FIG2_DIR, cbar_label=bafa1_cbar_label)

## 11.2 Autophagy machinery (SI3B)

In [ ]:
# SI3B, 1 x 3 units: BafA1 effect for the autophagy machinery, drawn as Figure 2G.
autophagy_dotplot_groups = [
    ("Aggrephagy receptors", [
        "SQSTM1",       # p62
        "CALCOCO2",     # NDP52
        "CALCOCO1",     # Golgiphagy / ER-phagy receptor
        "TAX1BP1",
        "NBR1",
        "OPTN",
    ]),
    ("Ferritino-phagy", [
        "NCOA4", "FTH1", "FTL",
    ]),
    ("Mitophagy receptors", [
        "BNIP3", "BNIP3L", "FUNDC1", "PGAM5",
        "PHB2",         # inner-membrane mitophagy receptor
        "BCL2L13",
        "PRKN", "PINK1",
    ]),
    ("ER-phagy receptors", [
        "RETREG1", "SEC62", "TEX264", "CCPG1", "ATL3",
    ]),
    ("Autophagosome formation", [
        "ATG5", "ATG7", "ATG3", "ATG16L1",
        "ATG12",        # conjugation system
        "MAP1LC3B",
        "GABARAP", "GABARAPL1", "GABARAPL2",
        "WIPI2", "ATG9A",
    ]),
    ("Initiation / signaling", [
        "RB1CC1", "ATG4B", "PIK3C3", "AMBRA1", "MTOR",
    ]),
    ("Lysosome / BafA1 target", [
        "ATP6V1A",      # V-ATPase V1, the direct BafA1 target
        "ATP6V0D1",     # V-ATPase V0
        "ATP6V1B2",
        "LAMP1",
        "LAMP2",
        "CTSD",
        "CTSB",
    ]),
    ("Aggregate clearance machinery", [
        "VCP", "UBE2N", "HDAC6", "BAG3",
        "HSPB8",        # CASA module with BAG3
        "STUB1",        # CHIP
        "UBQLN2",       # aggrephagy shuttle
        "PARK7",
    ]),
]

ttest_dotplot(autophagy_dotplot_groups, ttest_results, gene_map, bafa1_comparisons,
              bafa1_labels, "b", "BafA1_autophagy_dotplot", SI3_DIR,
              cbar_label=bafa1_cbar_label)

# 12. Aggregate composition (Figure 2F)

Candidates are proteins enriched in the excised aggregates: the top 15 by log2FC (FDR < 0.05, log2FC > 0) of each of three contrasts (EA / RC −BafA1, EA / RC +BafA1, BafA1 effect in EA), plus the proteins in the top 50 of at least two of them. The candidates are grouped by function by hand.

In [ ]:
# The three contrasts that define the candidates, and their names in the printouts
heatmap_comparisons = [
    "PureAggs_vs_RestCells_minusBafA1",
    "PureAggs_vs_RestCells_plusBafA1",
    "BafA1_effect_PFF_PureAggs",
]
heatmap_labels = [
    "Pure Aggs / Rest Cells (−BafA1)",
    "Pure Aggs / Rest Cells (+BafA1)",
    "BafA1 effect (Pure Aggs)",
]

In [ ]:
# Top 50 by log2FC (FDR < 0.05, log2FC > 0) of each contrast; the "robust"
# proteins are in at least two of the three lists.

top50_sets = {}
for comp, label in zip(heatmap_comparisons, heatmap_labels):
    res = ttest_results[comp].copy()
    top50 = res[(res["fdr"] < 0.05) & (res["log2fc"] > 0)].nlargest(50, "log2fc")
    top50_sets[label] = set(top50["protein"].tolist())
    print(f"{label}: {len(top50)} proteins")

# Keep proteins appearing in >= 2 lists
counts = Counter(p for s in top50_sets.values() for p in s)
robust = sorted([p for p, c in counts.items() if c >= 2])

print(f"\nUnion: {len(set().union(*top50_sets.values()))}")
print(f"In ≥2 lists: {len(robust)}")

# protein ID -> gene symbol, for the list in the next cell
name_map = adata_to_use.var["single_gene_name"].to_dict()

In [ ]:
# Candidates: the top 15 of each contrast plus the robust proteins, without
# duplicates. The next cell groups all of them.
top15_union = set()
for comp, label in zip(heatmap_comparisons, heatmap_labels):
    res = ttest_results[comp].copy()
    top15 = res[(res["fdr"] < 0.05) & (res["log2fc"] > 0)].nlargest(15, "log2fc")
    top15_union.update(top15["protein"].tolist())

combined = sorted(top15_union | set(robust), key=lambda p: name_map.get(p, p))
print(f"Top-15 unique: {len(top15_union)}")
print(f"Robust (≥2 lists): {len(robust)}")
print(f"Combined unique: {len(combined)}")
for i, p in enumerate(combined, 1):
    print(f"  {i:2d}. {name_map.get(p, p):15s}  [{p}]")

In [ ]:
# The candidates of the previous cell, grouped by function by hand
# (gene: description). The order of the groups and of the genes within each group
# is the row order of Figure 2F.

autophagy_receptors = {
    "SQSTM1":   "Sequestosome-1/p62",
    "TAX1BP1":  "Tax1-binding protein 1",
    "NBR1":     "Neighbor of BRCA1 (autophagy receptor)",
    "OPTN":     "Optineurin",
    "NCOA4":    "Nuclear receptor coactivator 4 (ferritinophagy)",
    "RTN3":     "Reticulon-3",
}

ubiquitin = {
    # Deubiquitinases (DUBs)
    "ZRANB1":   "Deubiquitinase ZRANB1",
    "ATXN3":    "Ataxin-3 (DUB, polyQ disease protein)",
    "MINDY1":   "MINDY1 (K48-specific DUB)",
    "MINDY3":   "MINDY3 (DUB)",
    "USP25":    "Ubiquitin-specific peptidase 25",
    # E3 ligases / adaptors
    "RNF115":   "E3 ubiquitin ligase RNF115",
    "RNF126":   "E3 ubiquitin ligase RNF126",
    "RNF149":   "E3 ubiquitin ligase RNF149",
    "NDFIP2":   "Nedd4 family-interacting protein 2",
}

erad_vcp = {
    "NPLOC4":   "NPL4 (VCP cofactor)",
    "UFD1":     "UFD1 (VCP cofactor)",
    "FAF1":     "FAS-associated factor 1",
}

golgi = {
    "FUT8":     "Fucosyltransferase 8",
    "ZFPL1":    "Zinc finger protein-like 1 (cis-Golgi)",
    "ERGIC2":   "ER-Golgi intermediate compartment 2",
    "TPRN":     "Taperin",
    "RAB34":    "Rab34 GTPase",
    "FKRP":     "Fukutin-related protein",
    "HS6ST2":   "Heparan sulfate 6-O-sulfotransferase 2",
}

endolysosomal = {
    "TM9SF1":   "Transmembrane 9 superfamily member 1",
    "TM9SF3":   "Transmembrane 9 superfamily member 3",
    "TMEM59":   "Transmembrane protein 59 (LC3-ATG16L1)",
    "ITM2C":    "Integral membrane protein 2C",
}

bafa1_substrates = {
    "ITM2B":    "Integral membrane protein 2B/BRI2",
    "APLP2":    "Amyloid-like protein 2",
    "APP":      "Amyloid precursor protein",
    "FAT1":     "Protocadherin FAT1",
    "LDLR":     "LDL receptor",
    "FZD2":     "Frizzled-2",
    "IFNGR1":   "IFN-γ receptor 1",
    "NEO1":     "Neogenin",
}

cell_division = {
    "CKAP2":    "Cytoskeleton-associated protein 2",
    "PSRC1":    "Proline/serine-rich coiled-coil 1",
    "MLF1":     "Myeloid leukemia factor 1",
    "CFAP410":  "Cilia/flagella-associated 410",
}

transporters = {
    "SLC20A1":  "Sodium-phosphate transporter SLC20A1",
    "SLC30A6":  "Zinc transporter ZnT6",
    "SLC35E1":  "Nucleotide-sugar transporter SLC35E1",
}

other = {
    "TRAFD1":   "TRAF-type zinc finger 1",
    "ZC3H3":    "Zinc finger CCCH-domain 3",
    "PLEKHA5":  "Pleckstrin homology domain A5",
}


# Groups in the row order of Figure 2F
enriched_groups = [
    ("Autophagy\nreceptors", autophagy_receptors),
    ("Ubiquitin", ubiquitin),
    ("ERAD / VCP", erad_vcp),
    ("Golgi", golgi),
    ("Endo-\nlysosomal", endolysosomal),
    ("BafA1\nsubstrates", bafa1_substrates),
    ("Cell division", cell_division),
    ("Transporters", transporters),
    ("Other", other),
]

In [ ]:
# Figure 2F, 1 x 3 units: the candidates in EA / RC (−BafA1), CW / CWO (−BafA1)
# and the BafA1 effect in EA, drawn as Figure 2G.
composition_groups = [(label, list(markers)) for label, markers in enriched_groups]

ttest_dotplot(composition_groups, ttest_results, gene_map,
              [heatmap_comparisons[0], "CellsWith_vs_CellsWithout_minusBafA1",
               heatmap_comparisons[2]],
              ["EA / RC", "CW / CWO", "+Baf / −Baf\n(EA)"],
              "f", "dotplot_aggregate_composition_CWvsCWO", FIG2_DIR,
              cbar_label="log2 fold change")

# 13. GSEA of the aggregate contrasts (SI3A)

The method of section 9.2 for four contrasts: EA / RC and CW / CWO, each without and with BafA1. Each contrast has its own decision file, made as in section 9.2 and carried over on re-runs; `gsea_dir/REVIEW_NOTES_aggregate_contrasts.md` records the reasoning of the AI review. The figure shares rows across the four contrasts by display label.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA of the aggregate contrasts [1/3]: EA / RC and CW / CWO, each −BafA1 and
# +BafA1. Ranking: Welch t-statistic, computed in the pipeline from adata_to_use.
# Null: gene permutation (prerank); the phenotype permutation is off for the
# reason given in section 9.2 [1/3].
# ═══════════════════════════════════════════════════════════════════════
# The terms of ARTIFACT_PATHWAYS in section 9.2 [1/3], under their own name.
AGG_ARTIFACT_PATHWAYS = frozenset({
    # Vascular / blood — no endothelium, no vessels, no plasma in the dish
    "GOBP_ENDOTHELIAL_CELL_MIGRATION",
    "GOBP_REGULATION_OF_BLOOD_CIRCULATION",
    "GOBP_REGULATION_OF_SYSTEMIC_ARTERIAL_BLOOD_PRESSURE",
    "GOBP_VASCULAR_PROCESS_IN_CIRCULATORY_SYSTEM",
    "REACTOME_CELL_SURFACE_INTERACTIONS_AT_THE_VASCULAR_WALL",
    "GOCC_BLOOD_MICROPARTICLE",

    # Immune — the named effector cell types are absent
    "GOBP_GRANULOCYTE_MIGRATION",
    "GOBP_POSITIVE_REGULATION_OF_IMMUNE_EFFECTOR_PROCESS",
    "GOBP_T_CELL_MEDIATED_IMMUNITY",

    # Embryonic tissue programmes — cannot occur in a monolayer
    "REACTOME_FORMATION_OF_PARAXIAL_MESODERM",
    "REACTOME_GASTRULATION",
    "REACTOME_SOMITOGENESIS",

    # Melanocytes — absent cell type
    "REACTOME_MITF_M_REGULATED_MELANOCYTE_DEVELOPMENT",
    "REACTOME_TRANSCRIPTIONAL_AND_POST_TRANSLATIONAL_REGULATION_OF_MITF_M_EXPRESSION_AND_ACTIVITY",

    # Cochlear hair cells — absent cell type
    "REACTOME_SENSORY_PROCESSING_OF_SOUND",
})

# All GSEA results go to gsea_dir, one subfolder per contrast.
cfg_agg = GSEAConfig(
    output_root=gsea_dir,
    artifact_pathways=AGG_ARTIFACT_PATHWAYS,
    n_review_clusters=40,
    run_phenotype=False,
    # Terms whose permutation null is constant and equal to the observed score
    # come back with NES = 1.0 and p = 1.0. These are not statistics and are
    # dropped; the number per contrast is printed below and goes in the legend.
    sentinel_action="drop",
    # plot_combined reads the style sizes for the tick labels, the axis labels and
    # the two keys, and derives its default canvas width from them.
    tick_fontsize=TICK_FS,
    label_fontsize=LABEL_FS,
    title_fontsize=TITLE_FS,
    cmap_colors=(DOWN_COLOR, "white", UP_COLOR),
)

gene_sets_agg = load_gene_sets(cfg_agg)   # cached in src/gsea_genesets_cache.parquet

# The conditions come from `comparisons` of the t-test cell, so the GSEA and the
# t-tests compare the same samples under the same name.
agg_contrasts = [
    "PureAggs_vs_RestCells_minusBafA1",
    "PureAggs_vs_RestCells_plusBafA1",
    "CellsWith_vs_CellsWithout_minusBafA1",
    "CellsWith_vs_CellsWithout_plusBafA1",
]
agg_labels = ["EA / RC −BafA1", "EA / RC +BafA1",
              "CW / CWO −BafA1", "CW / CWO +BafA1"]

_missing = [n for n in agg_contrasts if n not in comparisons]
assert not _missing, f"{_missing} not in `comparisons`; run the t-test cell first"

gsea_res_agg = {}
for name, label in zip(agg_contrasts, agg_labels):
    cond_a, cond_b = comparisons[name]          # (numerator, denominator)
    print(f"\n{'='*78}\n{name}   {label}\n  {cond_a}\n  vs {cond_b}\n{'='*78}")
    gsea_res_agg[name] = run_contrast(adata_to_use, biological, cond_a, cond_b,
                                      name, gene_sets_agg, cfg_agg)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA of the aggregate contrasts [2/3]: one decision file per contrast, as in
# section 9.2 [2/3].
# ═══════════════════════════════════════════════════════════════════════
AGG_REVIEW_PATHS = {}
for name in agg_contrasts:
    AGG_REVIEW_PATHS[name] = cfg_agg.contrast_dir(name) / "review.csv"
    assert AGG_REVIEW_PATHS[name].exists(), \
        f"decision file missing: {AGG_REVIEW_PATHS[name]}; check gsea_dir"
    build_review(gsea_res_agg[name], cfg_agg)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA of the aggregate contrasts [3/3]: check the decision files, then plot.
# SI3A, 2 x 3 units. Rows are display labels shared by the four contrasts; each
# cell shows that contrast's own result for the row's term. Cell states:
#   filled   FDR < 0.05 in this contrast
#   hollow   tested, FDR >= 0.05
#   grey x   outside this contrast's gene-set size window, not tested
# The size window counts the genes detected in each contrast, so a term can be
# tested in one contrast and not in another.
#
# cell_terms, anchor_rule and min_rows_per_contrast are set explicitly. Their
# defaults in gsea_pipeline ("purity", "coverage", 3) need a gene list for every
# display label in src/gsea_label_genes.yaml, which most labels of this figure
# do not have.
# ═══════════════════════════════════════════════════════════════════════
AGG_ROWS_UP, AGG_ROWS_DOWN = 20, 20
AGG_SIZE = (2, 3)                    # panel size in units (columns, rows)

# Dot area at the smallest and the largest detected set size, from printed
# diameters of 2.0 and 5.0 pt. The check below prints the largest dot against
# the row pitch.
AGG_DOT_AREA = (native_pt(2.0) ** 2, native_pt(5.0) ** 2)

# Display settings, set here so that a change needs a redraw and no new prerank.
# dataclasses.replace returns a new config. The line widths are the style values;
# plot_combined's defaults print below 0.25 pt on this canvas.
cfg_agg = dataclasses.replace(cfg_agg,
                              dot_area_min=AGG_DOT_AREA[0],
                              dot_area_max=AGG_DOT_AREA[1],
                              edge_lw_sig=MARKER_LW, edge_lw_ns=AXES_LW,
                              cross_lw=AXES_LW, grid_lw=THIN_LW,
                              key_edge_lw=MARKER_LW, key_marker_size=native_pt(3.0),
                              key_mew_sig=MARKER_LW, key_mew_ns=AXES_LW,
                              keys_position="corner", contrast_label_rotation=90)

for name, path in AGG_REVIEW_PATHS.items():
    rev = pd.read_csv(path, keep_default_na=False)
    assert_pass2_selection(rev)          # stops if a chosen representative would be lost
    report_undecided(rev, name)

# AGG_ROWS_UP and AGG_ROWS_DOWN set the rows per direction. More rows than the
# reviewed clusters provide would need a larger n_review_clusters in [1/3] and a
# review of the added clusters. save=True writes the row table (_grid.csv) into
# cfg.output_root; save_panel below writes the panel. The contrast labels are set
# on two lines: "EA / RC" above "−BafA1".
fig_agg, ax_agg = plot_combined(agg_contrasts, cfg_agg,
                                contrast_labels=["\n".join(l.rsplit(" ", 1))
                                                 for l in agg_labels],
                                rows_up=AGG_ROWS_UP, rows_down=AGG_ROWS_DOWN,
                                figsize=figsize(*AGG_SIZE), save=True,
                                save_stem="aggregate_contrasts_dotplot",
                                cell_terms="canonical",
                                anchor_rule="peak",
                                min_rows_per_contrast=0)

# ----------------------------------------------------------------- CHECKS
# Printed, not asserted: the panel is saved either way, and save_panel reports
# anything drawn past the canvas edge (legends, clipped row labels).
if fig_agg is not None:
    fig_agg.canvas.draw()
    _rows = len(ax_agg.get_yticklabels())
    _ax_w = ax_agg.get_window_extent().width / fig_agg.dpi
    print(f"CHECK plotting area {_ax_w:.1f} in wide on the canvas "
          f"({_ax_w / len(agg_contrasts):.1f} in per contrast column)")
    if _ax_w < 1.5 * len(agg_contrasts):
        print("  !! the row labels and the keys leave little room for the columns; "
              "widen AGG_SIZE")
    # Largest dot against the row pitch: dots that exceed it collide vertically.
    _pitch = ax_agg.get_window_extent().height / fig_agg.dpi * 72 / _rows
    _dmax = np.sqrt(AGG_DOT_AREA[1])
    print(f"CHECK largest dot {_dmax / DRAW_SCALE:.1f} pt across, "
          f"row pitch {_pitch / DRAW_SCALE:.1f} pt (printed)")
    if _dmax >= _pitch:
        print("  !! dots are taller than the row pitch; lower AGG_DOT_AREA[1]")

    save_panel(fig_agg, "a", "gsea_aggregate_contrasts", SI3_DIR)

# 14. BafA1-induced shift in aggregate enrichment

## 14.1 GSEA of the shift (SI3C)

The shift of a protein is log2FC(EA / RC, +BafA1) minus log2FC(EA / RC, −BafA1). Up: the protein is enriched in the aggregates more with BafA1 than without; down: less. GSEA runs prerank on this ranking, since no sample groups remain to permute. The decision file is made as in section 9.2 and carried over on re-runs; `gsea_dir/REVIEW_NOTES_bafa1_shift.md` records the reasoning of the AI review.

In [ ]:
# Per protein: mean_enrichment, the mean of the two EA / RC log2FCs (x axis of the
# MA plot), and bafa1_shift, +BafA1 minus −BafA1 (y axis of the MA plot and the
# GSEA ranking).

name_minus = "PureAggs_vs_RestCells_minusBafA1"
name_plus  = "PureAggs_vs_RestCells_plusBafA1"

scatter_df = ttest_results[name_minus][["protein", "log2fc"]].rename(columns={"log2fc": "minusBaf_log2fc"})
scatter_df = scatter_df.merge(
    ttest_results[name_plus][["protein", "log2fc"]].rename(columns={"log2fc": "plusBaf_log2fc"}),
    on="protein",
)
scatter_df = scatter_df.merge(
    adata_log.var[["single_gene_name"]], left_on="protein", right_index=True, how="left"
)
scatter_df["mean_enrichment"] = (scatter_df["minusBaf_log2fc"] + scatter_df["plusBaf_log2fc"]) / 2
scatter_df["bafa1_shift"]     = scatter_df["plusBaf_log2fc"] - scatter_df["minusBaf_log2fc"]

# Outliers beyond ±1.5 SD, coloured in the MA plot (not used by the GSEA)
mean_M = scatter_df["bafa1_shift"].mean()
sd_M   = scatter_df["bafa1_shift"].std()
upper  = mean_M + 1.5 * sd_M
lower  = mean_M - 1.5 * sd_M
scatter_df["plot_layer"] = "background"
scatter_df.loc[scatter_df["bafa1_shift"] > upper, "plot_layer"] = "up_outlier"
scatter_df.loc[scatter_df["bafa1_shift"] < lower, "plot_layer"] = "down_outlier"

print(f"Total proteins: {len(scatter_df)}")
print(f"  Up outliers (>1.5 SD):   {(scatter_df['plot_layer'] == 'up_outlier').sum()}")
print(f"  Down outliers (<-1.5 SD): {(scatter_df['plot_layer'] == 'down_outlier').sum()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA of the BafA1 shift [1/3]. The ranking is bafa1_shift, one number per
# protein from two contrasts, so no sample groups remain to permute:
# run_contrast_from_rank runs the prerank part only, and params.json records
# what was ranked.
# ═══════════════════════════════════════════════════════════════════════
SHIFT_NAME = "BafA1_shift_PureAggs_vs_RestCells"

# AGG_ARTIFACT_PATHWAYS comes from section 13 [1/3]; the system is the same.
assert "AGG_ARTIFACT_PATHWAYS" in globals(), \
    "run the GSEA [1/3] cell of section 13 first; it defines AGG_ARTIFACT_PATHWAYS"

# Same output folder as the other GSEA sections, own subfolder.
cfg_shift = GSEAConfig(
    output_root=gsea_dir,
    artifact_pathways=AGG_ARTIFACT_PATHWAYS,
    n_review_clusters=40,
    run_phenotype=False,        # required: a ranking carries no class vector
    sentinel_action="drop",     # disclose the dropped count in the legend
    tick_fontsize=TICK_FS,
    label_fontsize=LABEL_FS,
    title_fontsize=TITLE_FS,
    cmap_colors=(DOWN_COLOR, "white", UP_COLOR),
)

gene_sets_shift = load_gene_sets(cfg_shift)     # cached in src/gsea_genesets_cache.parquet
print(f"Gene sets: {gene_sets_shift['source'].nunique()} (Reactome + GO BP)")

# --- Ranking -----------------------------------------------------------
# One row per gene symbol: a symbol in several protein groups keeps the group
# with the highest mean log2 intensity across all samples. The abundance does not
# depend on the ranked statistic, so this choice cannot bias the enrichment.
_abund = pd.Series(np.nanmean(adata_to_use.X, axis=0), index=adata_to_use.var_names)
_r = (scatter_df.dropna(subset=["single_gene_name", "bafa1_shift"])
      .assign(_abund=lambda d: d["protein"].map(_abund))
      .sort_values("_abund", ascending=False)
      .drop_duplicates(subset=["single_gene_name"], keep="first")
      .set_index("single_gene_name")["bafa1_shift"].astype(float))
shift_rank = _r[np.isfinite(_r)].sort_values(ascending=False)

_n_dup = (scatter_df["single_gene_name"].notna().sum() - len(shift_rank))
print(f"Ranking: {len(shift_rank)} genes, {_n_dup} duplicate-symbol rows dropped "
      f"[{shift_rank.iloc[0]:+.2f} … {shift_rank.iloc[-1]:+.2f}]")

gsea_res_shift = run_contrast_from_rank(
    shift_rank, SHIFT_NAME, gene_sets_shift, cfg_shift,
    ranking_description=(
        "bafa1_shift = log2FC(Pure Aggregates / Rest Cells, +BafA1) − "
        "log2FC(Pure Aggregates / Rest Cells, −BafA1), Welch t-test log2FCs, "
        "symbols collapsed by highest mean log2 intensity"),
    extra_params={
        "design": "difference of two independent two-group contrasts; "
                  "no sample groups remain, so no phenotype null exists",
        "symbol_collapse": "highest mean log2 intensity across all samples",
        "n_proteins_in": int(len(scatter_df)),
    })

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA of the BafA1 shift [2/3]: the decision file, as in section 9.2 [2/3].
# ═══════════════════════════════════════════════════════════════════════
SHIFT_REVIEW_PATH = cfg_shift.contrast_dir(SHIFT_NAME) / "review.csv"
assert SHIFT_REVIEW_PATH.exists(), f"decision file missing: {SHIFT_REVIEW_PATH}; check gsea_dir"
_ = build_review(gsea_res_shift, cfg_shift)   # the returned table is in review.csv

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# GSEA of the BafA1 shift [3/3]: check the decision file, then plot. SI3C,
# 2 x 1 units. Single contrast: one row per cluster, labelled "Name (N)" with N
# the number of terms collapsed into it. Every row is FDR < 0.05, which the legend
# states in place of a significance key.
# The cell also builds selected_19, the plotted terms, for the MA plot.
# ═══════════════════════════════════════════════════════════════════════
SHIFT_SIZE = (2, 1)          # panel size in units (columns, rows)

# NES is the x axis, so there is no colourbar; the size key goes in the empty
# lower right of the axes; the collapse count is written "(9)".
# Largest dot = 0.8 of the row pitch. dot_area_min / dot_area_max set only the
# ratio, so the smallest dot is sqrt(300 / 900) = 0.58 of the largest.
cfg_shift = dataclasses.replace(cfg_shift, dot_fraction_of_row=0.8,
                                dot_area_min=300, dot_area_max=900,
                                single_edge_lw=MARKER_LW, single_zero_lw=AXES_LW,
                                key_edge_lw=MARKER_LW, keys_position="inside",
                                nes_colorbar=False, short_collapse_label=True)

rev_shift = pd.read_csv(SHIFT_REVIEW_PATH, keep_default_na=False)
assert_pass2_selection(rev_shift)        # stops if a chosen representative would be lost
report_undecided(rev_shift)

fig_shift, ax_shift = plot_contrast(SHIFT_NAME, cfg_shift, save=False,
                                    figsize=figsize(*SHIFT_SIZE), title="")
# Horizontal position of the size key, in fractions of the axes width: its lower
# right corner sits at KEY_X, minus matplotlib's small padding. 1.0 is the right
# axis edge; larger moves the key further right.
KEY_X = 1.03
ax_shift.get_legend().set_bbox_to_anchor((KEY_X, 0.0), transform=ax_shift.transAxes)

# ----------------------------------------------------------------- CHECKS
# Rows are clusters here, so two clusters with one label would draw two rows that
# a reader cannot tell apart.
_rows = [re.sub(r"\s+\(\d+( terms collapsed)?\)$", "", t.get_text())
         for t in ax_shift.get_yticklabels()]
_dups = sorted({r for r in _rows if _rows.count(r) > 1})
print(f"CHECK row labels drawn more than once: {_dups or 'none'}")

save_panel(fig_shift, "c", "gsea_bafa1_shift", SI3_DIR)

# --- Plotted terms for the MA plot -------------------------------------
# The MA-plot cells label one protein per plotted term, so they need the rows the
# figure drew: plot_contrast's selection, i.e. the representatives kept in
# review.csv, top_n_plot per direction by NES.
_reps = assert_pass2_selection(rev_shift).copy()
for _c in ("NES", "fdr", "cluster_size", "detected_size"):
    _reps[_c] = pd.to_numeric(_reps[_c], errors="coerce")
selected_19 = pd.concat([
    _reps[_reps["NES"] > 0].nlargest(cfg_shift.top_n_plot, "NES"),
    _reps[_reps["NES"] < 0].nsmallest(cfg_shift.top_n_plot, "NES"),
]).sort_values("NES").reset_index(drop=True)
selected_19 = selected_19.rename(columns={"term": "pathway"})
selected_19["pathway_clean"] = [
    str(r["display_label"]).strip() or r["pathway"].replace("_", " ").title()
    for _, r in selected_19.iterrows()
]

print(f"\nselected_19: {len(selected_19)} pathways for the MA plot "
      f"({int((selected_19['NES'] > 0).sum())} up, "
      f"{int((selected_19['NES'] < 0).sum())} down)")

## 14.2 MA plot (SI3D)

In [ ]:
# One labelled protein per plotted term: among the term's genes, the protein with
# the largest |bafa1_shift| in the term's direction that no stronger term has taken.

representative_labels = []
seen_proteins = set()

# Strongest |NES| first, so the strongest terms choose first
order = selected_19.reindex(selected_19["NES"].abs().sort_values(ascending=False).index)

for _, row in order.iterrows():
    pathway = row["pathway"]
    nes     = row["NES"]
    members = set(gene_sets_shift[gene_sets_shift["source"] == pathway]["target"].values)
    direction = "UP" if nes > 0 else "DOWN"

    candidates = scatter_df[
        scatter_df["single_gene_name"].isin(members)
        & (scatter_df["bafa1_shift"] > 0 if nes > 0 else scatter_df["bafa1_shift"] < 0)
        & ~scatter_df["protein"].isin(seen_proteins)
    ]
    if candidates.empty:
        continue

    best = candidates.iloc[candidates["bafa1_shift"].abs().argmax()]
    seen_proteins.add(best["protein"])
    representative_labels.append({
        "protein":     best["protein"],
        "gene":        best["single_gene_name"],
        "term":        row["pathway_clean"],
        "direction":   direction,
        "bafa1_shift": best["bafa1_shift"],
        "NES":         nes,
    })

print(f"\nRepresentative labels: {len(representative_labels)}\n")
for r in representative_labels:
    arrow = "↑" if r["direction"] == "UP" else "↓"
    print(f"  {arrow} {r['gene']:15s}  shift={r['bafa1_shift']:+.2f}  NES={r['NES']:+.2f}  ({r['term']})")

representative_labels_df = pd.DataFrame(representative_labels)

In [ ]:
# SI3D, 1 x 1 unit: MA plot of the BafA1 shift against the mean aggregate
# enrichment, with the representative protein of each plotted GSEA term. The
# shift is a direction of change, so up and down take UP_COLOR and DOWN_COLOR,
# as in the volcano and the dot plots.
SHIFT_UP, SHIFT_DOWN = UP_COLOR, DOWN_COLOR
REP_DOT = native_pt(3.0)     # representative proteins, printed diameter
BG_DOT  = native_pt(1.0)     # background cloud
OUT_DOT = native_pt(1.4)     # outliers that are not representatives

# Which proteins are representatives (previous cell)
rep_proteins = [r["protein"] for r in representative_labels]
scatter_df["is_representative"] = scatter_df["protein"].isin(rep_proteins)

# One shade per term from DIRECTION_CMAP: the red half for up, the blue half for
# down, the scale of the GSEA panel. The shades stop short of the white centre of
# the map, so that the palest stays visible.
up_reps   = [r for r in representative_labels if r["direction"] == "UP"]
down_reps = [r for r in representative_labels if r["direction"] == "DOWN"]
up_palette   = DIRECTION_CMAP(np.linspace(0.72, 1.0, max(len(up_reps), 1)))
down_palette = DIRECTION_CMAP(np.linspace(0.28, 0.0, max(len(down_reps), 1)))
protein_colors = {}
for rep, color in zip(up_reps,   up_palette):   protein_colors[rep["protein"]] = color
for rep, color in zip(down_reps, down_palette): protein_colors[rep["protein"]] = color

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

# Background
bg = scatter_df[scatter_df["plot_layer"] == "background"]
ax.scatter(bg["mean_enrichment"], bg["bafa1_shift"], s=BG_DOT ** 2, alpha=0.45,
           c=BG_COLOR, linewidths=0, zorder=1)

# Up and down outliers that are not representatives (those get their own marker)
up_out = scatter_df[(scatter_df["plot_layer"] == "up_outlier") & ~scatter_df["is_representative"]]
ax.scatter(up_out["mean_enrichment"], up_out["bafa1_shift"],
           s=OUT_DOT ** 2, alpha=0.45, c=SHIFT_UP, linewidths=0, zorder=2)
down_out = scatter_df[(scatter_df["plot_layer"] == "down_outlier") & ~scatter_df["is_representative"]]
ax.scatter(down_out["mean_enrichment"], down_out["bafa1_shift"],
           s=OUT_DOT ** 2, alpha=0.45, c=SHIFT_DOWN, linewidths=0, zorder=2)

# Representative proteins, each in the shade of its term
for rep in representative_labels:
    row = scatter_df[scatter_df["protein"] == rep["protein"]]
    if row.empty:
        continue
    row = row.iloc[0]
    ax.scatter(row["mean_enrichment"], row["bafa1_shift"],
               s=REP_DOT ** 2, c=[protein_colors[rep["protein"]]],
               edgecolors="black", linewidths=MARKER_LW, zorder=5)

# Reference lines: zero shift, the ±1.5 SD outlier cut-offs, zero enrichment
ax.axhline(0,     color=REF_COLOR,  linewidth=THIN_LW, linestyle="--", zorder=0)
ax.axhline(upper, color=SHIFT_UP,   linewidth=THIN_LW, linestyle=":", alpha=0.8, zorder=0)
ax.axhline(lower, color=SHIFT_DOWN, linewidth=THIN_LW, linestyle=":", alpha=0.8, zorder=0)
ax.axvline(0,     color=BG_COLOR,   linewidth=THIN_LW, zorder=0)

# The key goes below the panel: the upper left of the axes holds the up-shifted,
# low-enrichment proteins and their labels. One column: the two entries side by
# side are wider than the 42 mm panel.
legend_handles = [
    Line2D([0], [0], marker="o", ls="none", markerfacecolor=SHIFT_UP, markeredgewidth=0,
           markersize=native_pt(2.4), label="Up outlier (> 1.5 SD)"),
    Line2D([0], [0], marker="o", ls="none", markerfacecolor=SHIFT_DOWN, markeredgewidth=0,
           markersize=native_pt(2.4), label="Down outlier (< −1.5 SD)"),
]
_leg = fig.legend(handles=legend_handles, loc="outside lower center", ncol=1,
                  frameon=False)
# h_pad is the constrained-layout gap around the axes, in inches, and it sets
# the distance between the x axis label and the legend strip below it.
fig.get_layout_engine().set(h_pad=0.22)

label_axes(ax,
           xlabel="Mean aggregate enrichment\n(log2FC)",
           ylabel="BafA1-induced shift\n(Δlog2FC)")

# Labels: gene symbols of the representatives, each at a fixed position chosen by
# hand (centre of the label, data coordinates). Two pairs of dots cover each other
# and share a label: RBBP5 and CCNH, H2BC12 and H2BC1. adjust_text, used before,
# placed the labels differently whenever the text widths changed by a fraction of
# a pixel, and onto the axis lines.
LABELS = {
    # label:         (genes it names,       x,     y)
    "TAPBP":         (["TAPBP"],           0.35,  1.85),
    "CLN3":          (["CLN3"],            2.05,  1.75),
    "CEP72":         (["CEP72"],           2.00,  1.00),
    "H2BC26":        (["H2BC26"],         -1.88,  0.03),
    "RBBP5/CCNH":    (["RBBP5", "CCNH"],   0.07,  0.08),
    "SF3B5":         (["SF3B5"],           2.03,  0.02),
    "PRMT9":         (["PRMT9"],           1.55, -0.66),
    "LETMD1":        (["LETMD1"],          1.45, -1.32),
    "CHCHD7":        (["CHCHD7"],          1.30, -1.97),
    "RPP40":         (["RPP40"],          -0.40, -1.35),
    "H2BC12/H2BC1":  (["H2BC12", "H2BC1"], -1.27, -2.17),
}
labeled = scatter_df[scatter_df["is_representative"]].set_index("single_gene_name")
_named = sorted(g for genes, _, _ in LABELS.values() for g in genes)
assert _named == sorted(labeled.index), \
    f"LABELS must name each representative once: {_named} vs {sorted(labeled.index)}"

texts = [ax.text(x, y, name, ha="center", va="center", fontsize=ANNOT_FS,
                 bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                           edgecolor="none", alpha=0.85))
         for name, (_, x, y) in LABELS.items()]

# Leaders run from the point of the label box nearest to the dot (for a shared
# label, the point between its two dots) and end at the dot edge.
fig.canvas.draw()   # the label boxes are measured in display coordinates
for t, (genes, _, _) in zip(texts, LABELS.values()):
    dot = labeled.loc[genes, ["mean_enrichment", "bafa1_shift"]].mean().to_numpy()
    b = t.get_bbox_patch().get_window_extent()
    px, py = ax.transData.transform(dot)
    start = ax.transData.inverted().transform((np.clip(px, b.x0, b.x1), np.clip(py, b.y0, b.y1)))
    ax.annotate("", xy=dot, xytext=start,
                arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW,
                                shrinkA=0, shrinkB=REP_DOT / 2))

# ----------------------------------------------------------------- CHECKS
fig.canvas.draw()
_lb = _leg.get_window_extent()
_ab = ax.get_window_extent()
_boxes = [t.get_bbox_patch().get_window_extent() for t in texts]
_hit = [t.get_text() for t, b in zip(texts, _boxes) if _lb.overlaps(b)]
_out = [t.get_text() for t, b in zip(texts, _boxes)
        if b.x0 < _ab.x0 or b.x1 > _ab.x1 or b.y0 < _ab.y0 or b.y1 > _ab.y1]
_pairs = [(texts[i].get_text(), texts[j].get_text())
          for i in range(len(texts)) for j in range(i + 1, len(texts))
          if _boxes[i].overlaps(_boxes[j])]
print(f"CHECK labels under the legend (must be none): {_hit or 'none'}")
print(f"CHECK labels past the axes (must be none): {_out or 'none'}")
print(f"CHECK overlapping labels (must be none): {_pairs or 'none'}")
print(f"CHECK {len(texts)} labels name {len(_named)} of {len(representative_labels)} representatives")

save_panel(fig, "d", "MA_plot_bafa1_shift", SI3_DIR)

In [ ]:
# Source data of the MA plot (SI3D): one row per protein, with the two log2 fold
# changes it is built from, its x (mean enrichment) and y (BafA1 shift), its
# layer, and for the representatives the GSEA term they label. The ±1.5 SD
# outlier cut-offs are the two constant columns at the end.
ma_export = scatter_df[["protein", "single_gene_name", "minusBaf_log2fc", "plusBaf_log2fc",
                        "mean_enrichment", "bafa1_shift", "plot_layer",
                        "is_representative"]].merge(
    representative_labels_df[["protein", "term", "direction", "NES"]].rename(
        columns={"term": "representative_term", "direction": "representative_direction",
                 "NES": "representative_NES"}),
    on="protein", how="left")
ma_export["outlier_lower"] = lower
ma_export["outlier_upper"] = upper
ma_export = ma_export.sort_values("bafa1_shift", ascending=False)

_path = f"{output_dir}/{today}_MA_plot_bafa1_shift.csv"
ma_export.to_csv(_path, index=False)
print(f"written: {_path}  ({len(ma_export)} proteins, "
      f"{int(ma_export['is_representative'].sum())} representatives, "
      f"cut-offs {lower:+.3f} / {upper:+.3f})")

# 15. Ferritinophagy (SI3E)

In [ ]:
# SI3E, 1 x 3 units: receptor, cargo and iron machinery across the five BafA1
# effects and CW against CWO (−BafA1), drawn as Figure 2G.
ferritin_comparisons = bafa1_comparisons + ["CellsWith_vs_CellsWithout_minusBafA1"]
ferritin_labels = ["Baf PBS", "Baf CWO", "Baf CW", "Baf EA", "Baf RC", "CW vs CWO (−Baf)"]

ferritin_groups = [
    ("Receptor", [
        "NCOA4",            # the ferritinophagy receptor
    ]),
    ("Cargo", [
        "FTH1",             # ferritin heavy chain
        "FTL",              # ferritin light chain
        "FTMT",             # mitochondrial ferritin
    ]),
    ("IRPs", [
        "IREB2",            # IRP2 — translational regulator of FTH1 / FTL / TFRC
        "ACO1",             # IRP1 — iron-sensing aconitase, IRE-binding when iron-low
    ]),
    ("Fe transport", [
        "TFRC",             # transferrin receptor 1 — iron uptake
        "TFR2",             # transferrin receptor 2 — alternative uptake
        "TF",               # transferrin — iron carrier, made in the liver
        "SLC11A2",          # DMT1 — endosomal Fe2+ release
        "SLC40A1",          # ferroportin — cellular Fe export
    ]),
    ("Fe chaperones", [
        "PCBP1",            # iron chaperone — delivers Fe to ferritin (FTH1)
        "PCBP2",            # iron chaperone — delivers Fe to ferritin
        "PCBP3",            # related, less characterized
        "PCBP4",            # related, less characterized
        "BDH2",             # siderophore-like iron chaperone supporting ferritin loading
        "CISD1",            # mitochondrial outer-membrane Fe-S (mitoNEET)
        "CISD2",            # ER / mito-associated Fe-S
    ]),
    ("E2 / E3", [
        "HERC2",            # NCOA4 E3 — degrades receptor when iron is high
        "UBE2O",            # NCOA4 E2
        "FBXL5",            # IREB2 / ACO1 F-box E3 — iron-responsive degradation
    ]),
    ("CIA (Fe-S)", [
        "CIAO1",            # cytosolic Fe-S assembly
        "MMS19",            # cytosolic Fe-S assembly
        "IBA57",            # Fe-S assembly factor
        "NUBPL",            # mitochondrial complex I Fe-S assembly
    ]),
    ("ATG8 family", [
        "MAP1LC3A", "MAP1LC3B", "MAP1LC3C",
        "GABARAP", "GABARAPL1", "GABARAPL2",
    ]),
    ("Lysosome", [
        "LAMP1", "LAMP2",
        "STEAP1",           # ferrireductase Fe3+ → Fe2+
        "STEAP2",           # ferrireductase
        "STEAP3",           # endosomal / lysosomal ferrireductase (also TSAP6, exosome-related)
        "STEAP4",           # ferrireductase
    ]),
    ("V-ATPase (BafA1 target)", [
        "ATP6V0A1",         # V0 subunit a1 — lysosomal
        "ATP6V0D1",         # V0 subunit d1
        "ATP6V1A",          # V1 subunit A — catalytic
        "ATP6V1B2",         # V1 subunit B2
        "ATP6V1H",          # V1 subunit H
    ]),
    ("Heme & redox", [
        "HMOX1",            # heme oxygenase 1 — iron release from heme
        "HMOX2",            # heme oxygenase 2 — constitutive
        "HIF1A",            # hypoxia-inducible factor 1α, hypoxia and iron signalling
        "EPAS1",            # HIF-2alpha — official symbol is EPAS1, not HIF2A
        "NFE2L2",           # NRF2 — master antioxidant + iron stress TF
        "KEAP1",            # NRF2 negative regulator
    ]),
]

# The dashed line separates the five BafA1 effects from CW against CWO
ttest_dotplot(ferritin_groups, ttest_results, gene_map, ferritin_comparisons,
              ferritin_labels, "e", "ferritinophagy_dotplot", SI3_DIR, size=(1, 3),
              separator_before=5, cbar_label="log2 FC")

# 16. Figure assembly

## 16.1 Figure 2

In [ ]:
# Figure 2 from the saved panels. Layout: one letter per grid unit, "." = empty.
figure_dir = FIG2_DIR
figure_name = f"{today}_Figure2_Autophagy"
layout = "a b d d \nc e f g\n. e f g\n. e f g"

compose_figure(
    folder=figure_dir,
    layout=layout,
    output=f"{figure_dir}/{figure_name}.svg",
    show_box=True,
)

## 16.2 SI2 and SI3

In [ ]:
# SI2 from the saved panels
figure_dir = SI2_DIR
figure_name = f"{today}_SI2_HEK_QC_AGG"
layout = "a a a d\nb b c c\ne e c c\nf f . ."

compose_figure(
    folder=figure_dir,
    layout=layout,
    output=f"{figure_dir}/{figure_name}.svg",
    show_box=True,
)

In [ ]:
# SI3 from the saved panels. Panel f is the microscopy image, an SVG made outside
# this notebook and not part of the repository. compose_figure looks for a file
# f_*.svg in SI3_DIR and stops with FileNotFoundError when there is none. Put the
# image there, or replace "f" by "." in the layout and delete "f" from panel_scale
# to assemble the other panels.
figure_dir = SI3_DIR
figure_name = f"{today}_SI3_HEK_BafA1"
layout = "a a b d\na a b e\na a b e\n c c . e"

compose_figure(
    folder=figure_dir,
    layout=layout,
    output=f"{figure_dir}/{figure_name}.svg",
    # panel_scale={"f": 1.0},             # microscopy image: not a save_panel panel, fill its column
    gap_x_mm=4,                         # horizontal gap only; the vertical gap stays 3 mm
    panel_offset_mm={"d": (1.5, 1.0)},  # move d's plot 1.5 mm right and 1 mm down; its letter stays
    show_box=True,
)